<a href="https://colab.research.google.com/github/gharsallahcharfeddine49-jpg/AlphaFold-Fusion-Premium/blob/main/AlphaFold_Fusion_Premium.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# AlphaFold Fusion Premium — All-in-One (AF Fidelity ↑, pLDDT/Identity, Monomer/Multimer, AFDB-first, Reliable 3D)
# Version: 2025-12-10 (improved: cache reuse, MSA cap, auto-speed optimize, robust pLDDT palettes, US-align pane, identity 3D)
# Includes: Robust A3M, PAE, API backoff, CIF label_seq_id toggle, pydantic<2 fallback, UniProt/AFDB link enrichment

import os, sys, subprocess, time, shutil
from pathlib import Path

print("🧬 AlphaFold Fusion Premium — AF fidelity ↑ (MSA/Multimer/Recycles) + pLDDT + Identity + AFDB-first + Reliable 3D")
print("=" * 80)

# Kill previous UI instances
os.system("pkill -f streamlit 2>/dev/null || true")
os.system("pkill -f cloudflared 2>/dev/null || true")
time.sleep(1)

# PIP environment
os.environ["PIP_DEFAULT_TIMEOUT"] = "900"
os.environ["PIP_PROGRESS_BAR"] = "off"
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"

def pip_install(args, timeout=2400, quiet=True):
    cmd = [sys.executable, "-m", "pip", "install"]
    if quiet:
        cmd.append("-q")
    cmd += ["--no-cache-dir"] + args
    print("    → pip", " ".join(args))
    return subprocess.run(cmd, timeout=timeout).returncode == 0

# Base tools
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"], check=False)

# ColabFold (with pydantic<2 fallback)
print("  → colabfold[alphafold]")
ok_cf = pip_install(["colabfold[alphafold] @ https://codeload.github.com/sokrypton/ColabFold/zip/refs/heads/main"], timeout=2400, quiet=False)
if not ok_cf:
    ok_cf = pip_install(["colabfold[alphafold]==1.5.5"], quiet=False)
if not ok_cf:
    ok_cf = pip_install(["pydantic<2", "colabfold[alphafold]==1.5.4"], quiet=False) or pip_install(["colabfold[alphafold]==1.5.4"], quiet=False)
if not ok_cf:
    raise RuntimeError("colabfold[alphafold] installation failed.")

# Scientific + UI deps
pip_install(["numpy==1.26.4"], quiet=False)
pip_install(["pandas>=2,<3"], quiet=False)
pip_install(["tensorflow==2.18.*", "protobuf>=4.25,<6"], quiet=False)  # AMBER optional
for pkg in [
    "streamlit==1.28.0",
    "plotly==5.17.0",
    "py3Dmol==2.1.0",
    "biopython>=1.83,<2",
    "Pillow==10.1.0",
    "psutil==5.9.8",
    "gemmi==0.6.6"
]:
    pip_install([pkg])

# JAX: consistent installation (GPU if available, plugin auto-match)
def install_jax_gpu():
    import subprocess, sys, re, shutil
    def _run(cmd): return subprocess.run(cmd, check=False, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    def _pip(args):
        print("    → pip", " ".join(args))
        return _run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "-q"] + args)
    _run([sys.executable, "-m", "pip", "uninstall", "-y", "jax-cuda12-plugin", "jaxlib", "jax"])
    _pip(["jax>=0.5.0,<0.6", "jaxlib>=0.5.0,<0.6"])
    has_gpu = shutil.which("nvidia-smi") is not None
    if has_gpu:
        try:
            import jaxlib as _jl
            jl_ver = getattr(_jl, "__version__", None) or _run([sys.executable,"-c","import jaxlib;print(jaxlib.__version__)"]).stdout.strip()
            print(f"🔧 detected jaxlib: {jl_ver}")
            if jl_ver and re.match(r"^\d+\.\d+\.\d+$", jl_ver):
                r = _pip([f"jax-cuda12-plugin=={jl_ver}"])
                if r.returncode != 0:
                    x, y, _ = jl_ver.split("."); alt = f"{x}.{y}.0"
                    print(f"⚠️ plugin {jl_ver} unavailable → trying {alt}")
                    _pip([f"jax-cuda12-plugin=={alt}"])
            else:
                _pip(["jax-cuda12-plugin>=0.5.0,<0.6"])
        except Exception as e:
            print("⚠️ CUDA12 plugin not finalized:", e)
    else:
        _run([sys.executable, "-m", "pip", "uninstall", "-y", "jax-cuda12-plugin"])
    try:
        import jax
        print(f"✅ JAX ready — backend: {jax.default_backend()} | devices: {jax.devices()}")
    except Exception as e:
        print("⚠️ JAX verification:", e)

install_jax_gpu()

# Check core libs (without importing TF)
try:
    import numpy as _np, pandas as _pd
    print(f"✅ Versions: numpy={_np.__version__} | pandas={_pd.__version__}")
except Exception as e:
    print("⚠️ NumPy/Pandas import:", e)

print("✅ Dependencies installed")
print("=" * 80)

# ---------------------- Streamlit Application ----------------------
APP = []
APP.append(r'''
import os, sys, subprocess, time, re, json, shutil, urllib.request, tempfile, secrets, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import streamlit as st
import plotly.graph_objects as go
import plotly.express as px
import py3Dmol, psutil, gemmi

from Bio.PDB import PDBIO
from Bio.PDB.MMCIFParser import MMCIFParser

# ---------- Page config ----------
st.set_page_config(page_title="AlphaFold Fusion Premium", page_icon="🧬", layout="wide", initial_sidebar_state="expanded")

BASE = Path("/content") if Path("/content").exists() else Path.cwd()
RESULTS_DIR = BASE / "alphafold_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = BASE / "alphafold_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def new_run_root():
    run_id = time.strftime("%Y%m%d-%H%M%S") + "-" + secrets.token_hex(3)
    root = RESULTS_DIR / f"run_{run_id}"
    root.mkdir(parents=True, exist_ok=True)
    st.session_state["_run_root"] = str(root)
    return root

def current_run_root():
    p = st.session_state.get("_run_root")
    return Path(p) if p else RESULTS_DIR

# ---------- CSS ----------
st.markdown("""
<style>
:root { --primary:#4361ee; --secondary:#3a0ca3; --accent:#7209b7; --success:#4cc9f0; --warning:#f72585; --light:#f8f9fa; --dark:#212529; }
.main-header { font-size:3rem; background:linear-gradient(135deg,#4361ee 0%,#3a0ca3 100%); -webkit-background-clip:text; -webkit-text-fill-color:transparent; text-align:center; margin-bottom:1rem; font-weight:800; }
.sub-header { font-size:1.6rem; color:#3a0ca3; margin:1rem 0 .6rem 0; border-bottom:3px solid #4cc9f0; }
.feature-card { padding:1rem; border-radius:1rem; background:#fff; box-shadow:0 10px 25px rgba(0,0,0,.08); margin:1rem 0; }
.info-card { background:linear-gradient(135deg,#f8f9fa 0%,#e9ecef 100%); padding:1rem; border-radius:1rem; border-left:5px solid #4361ee; margin:1rem 0; }
.stButton>button { background:linear-gradient(135deg,#4361ee 0%,#3a0ca3 100%); color:#fff; border:none; padding:.6rem 1.2rem; border-radius:50px; font-weight:600; width:100%; }
.viewer-container { border-radius:1rem; overflow:hidden; box-shadow:0 10px 25px rgba(0,0,0,.1); margin:1rem 0; }
</style>
""", unsafe_allow_html=True)

st.markdown('<div class="main-header">🧬 AlphaFold Fusion Premium</div>', unsafe_allow_html=True)
st.markdown('<p style="text-align:center;color:#666;margin-bottom:1rem;">AF Fidelity ↑ (MSA/Multimer/Recycles) + pLDDT/Identity + AFDB + Reliable 3D</p>', unsafe_allow_html=True)

# ---------- Session ----------
ss = st.session_state
for k, v in {"results":{}, "job_dirs":{}, "fasta_paths":{}, "fasta_text":"", "aln_import":{},
            "_uniprot_name_cache":{}, "_acc_to_uniprot_cache":{}, "_afdb_cache":{}, "_afdb_used":{}}.items():
    ss.setdefault(k, v)

# ---------- Sequence utils ----------
def clean_sequence_advanced(seq: str) -> str:
    seq = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY:]", "", (seq or "").upper())
    seq = re.sub(r":+", ":", seq)
    return seq.strip(":")

def detect_complex(seq: str) -> bool:
    return ":" in (seq or "")

def seq_length_total(seq: str) -> int:
    s = clean_sequence_advanced(seq)
    return sum(len(part) for part in s.split(":") if part)

def make_safe_basename(name: str, seq: str | None = None, maxlen: int = 80) -> str:
    """
    Filesystem-safe basename with hash suffix to guarantee uniqueness.
    """
    base = re.sub(r'[^A-Za-z0-9._-]+', '_', (name or 'sequence'))
    base = base.strip("._-") or "sequence"
    import hashlib as _hl
    h = _hl.sha1(((name or '') + '|' + (seq or '')).encode()).hexdigest()[:10]
    keep = max(8, maxlen - len(h) - 1)
    base = base[:keep]
    return f"{base}_{h}"

# ---------- Viewer helpers ----------
def viewer_key(prefix, path=None, style=None, scheme=None, mono=None, chain=None, first=None):
    p = Path(path).name if path else "mem"
    try:
        mtime = os.path.getmtime(path) if path and os.path.exists(path) else 0
    except Exception:
        mtime = 0
    payload = f"{p}|{mtime}|{style}|{scheme}|{mono}|{chain}|{first}"
    return f"{prefix}::{hashlib.md5(payload.encode()).hexdigest()}"

def apply_viewer_stamp(html: str, stamp: str) -> str:
    try:
        return f"<!-- {stamp} -->\n{html}"
    except Exception:
        return html

def patch_py3dmol_html(doc: str) -> str:
    try:
        import re as _re
        s = doc or ""
        s = _re.sub(
            r'src="https?://[^"]*3Dmol[^"]*\.js"',
            'src="https://cdn.jsdelivr.net/npm/3dmol@2.0.4/build/3Dmol-min.js" onerror="this.onerror=null;this.src=\'https://unpkg.com/3dmol/build/3Dmol-min.js\';"',
            s, count=1
        )
        return s
    except Exception:
        return doc

def enhance_py3dmol_html(doc: str) -> str:
    try:
        if not doc: return doc
        return doc
    except Exception:
        return doc

# ---------- UniProt + InterPro (with API backoff) ----------
@st.cache_data(show_spinner=False, ttl=86400)
def fetch_uniprot_json(acc: str, timeout: int = 12, retries: int = 2, backoff: float = 0.5):
    try_acc = (acc or "").strip()
    if not try_acc:
        return None
    for attempt in range(max(1, retries)):
        try:
            url = f"https://rest.uniprot.org/uniprotkb/{try_acc}.json"
            with urllib.request.urlopen(url, timeout=timeout) as r:
                return json.loads(r.read().decode("utf-8","ignore"))
        except Exception:
            time.sleep(backoff * (attempt + 1))
    return None

def domains_from_uniprot_json(j):
    segs = []
    try:
        feats = (j or {}).get("features") or []
        keep = {"Domain","Repeat","Region","Coiled coil","Zinc finger"}
        k = 1
        for f in feats:
            t = f.get("type")
            if t not in keep:
                continue
            loc = f.get("location") or {}
            beg = ((loc.get("start") or {}).get("value"))
            end = ((loc.get("end") or {}).get("value"))
            if beg is None or end is None:
                continue
            try:
                beg = int(beg); end = int(end)
            except Exception:
                continue
            lab = f.get("description") or t or f"Domain {k}"
            segs.append({"start": beg, "end": end, "label": lab, "type": t}); k += 1
    except Exception:
        pass
    return sorted(segs, key=lambda s: (s["start"], s["end"]))

@st.cache_data(show_spinner=False, ttl=86400)
def fetch_interpro_json(acc: str, timeout: int = 12, max_pages: int = 2, retries: int = 2, backoff: float = 0.5):
    try_acc = (acc or "").strip()
    if not try_acc:
        return None
    try:
        headers = {"Accept": "application/json"}
        base = f"https://www.ebi.ac.uk/interpro/api/protein/uniprot/{try_acc}"
        url = base
        all_results = []
        for _ in range(max_pages):
            ok = False
            for attempt in range(max(1, retries)):
                try:
                    req = urllib.request.Request(url, headers=headers)
                    with urllib.request.urlopen(req, timeout=timeout) as r:
                        j = json.loads(r.read().decode("utf-8","ignore"))
                    ok = True
                    break
                except Exception:
                    time.sleep(backoff * (attempt + 1))
            if not ok:
                break
            res = j.get("results") or []
            all_results.extend(res)
            next_url = j.get("next") or j.get("_links", {}).get("next", {}).get("href")
            if not next_url:
                break
            url = next_url
        return {"results": all_results}
    except Exception:
        return None

def interpro_segments_from_json(j):
    segs = []
    try:
        results = (j or {}).get("results") or []
        keep_types = {"domain","repeat","homologous superfamily","family"}
        for r in results:
            entry = r.get("entry") or {}
            etype = (entry.get("type") or entry.get("type_name") or "").strip().lower()
            if etype not in keep_types:
                continue
            acc = entry.get("accession") or entry.get("acc") or ""
            name = entry.get("name") or entry.get("short_name") or etype.title()
            label = f"{acc} {name}".strip()
            locations = r.get("locations") or []
            for loc in locations:
                frags = loc.get("fragments") or []
                for fr in frags:
                    try:
                        beg = int(fr.get("start")); end = int(fr.get("end"))
                        if beg is not None and end is not None and beg <= end:
                            segs.append({"start":beg,"end":end,"label":label,"type":etype.title()})
                    except Exception:
                        continue
    except Exception:
        pass
    uniq = []; seen = set()
    for s in segs:
        key = (s["start"], s["end"], s["label"])
        if key in seen:
            continue
        seen.add(key); uniq.append(s)
    return sorted(uniq, key=lambda x: (x["start"], x["end"]))

def _domain_color(i):
    palette = ["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b","#e377c2","#7f7f7f","#bcbd22","#17becf"]
    return palette[i % len(palette)]

def domain_legend_html(segs: list):
    items = []
    for i, s in enumerate(segs):
        color = _domain_color(i)
        lab = f'{s.get("label","Domain")} ({s.get("start","?")}-{s.get("end","?")})'
        items.append(f'<div style="display:flex;align-items:center;margin:4px 0;"><span style="display:inline-block;width:14px;height:14px;background:{color};border-radius:2px;margin-right:8px;"></span>{lab}</div>')
    return '<div style="border:1px solid #e5e7eb;border-radius:10px;padding:10px;background:#fff;">' + "".join(items) + "</div>"

# ---------- UniProt ACC utils + link enrichment ----------
def _is_uniprot_acc(s: str) -> bool:
    import re as _re
    if not s:
        return False
    t = s.strip().upper()
    return bool(_re.match(r"^[A-Z0-9]{6,10}(?:-\d+)?$", t))

def _extract_uniprot_acc_from_any(s: str) -> str | None:
    import re as _re
    if not s:
        return None
    raw = str(s).strip()
    for tok in _re.split(r"[|\s,;/]+", raw):
        if not tok:
            continue
        u = tok.strip().upper()
        if _is_uniprot_acc(u):
            return u
    m = _re.search(r"UniRef\d+_([A-Z0-9]{6,10}(?:-\d+)?)", raw, flags=_re.I)
    if m:
        cand = m.group(1).upper()
        if _is_uniprot_acc(cand):
            return cand
    return None

def annotate_df_with_uniprot_info(df, acc_col="Accession"):
    import pandas as _pd
    if df is None or df.empty or acc_col not in df.columns:
        return df
    out = df.copy()
    uni = []
    afdb = []
    for val in out[acc_col].astype(str).tolist():
        acc_canon = _extract_uniprot_acc_from_any(val)
        if acc_canon:
            uni.append(f"https://www.uniprot.org/uniprotkb/{acc_canon}")
            afdb.append(f"https://alphafold.ebi.ac.uk/entry/{acc_canon.split('-')[0]}")
        else:
            uni.append(None)
            afdb.append(None)
    out["UniProt_URL"] = uni
    out["AFDB_URL"] = afdb
    return out

def find_seq_name_by_model_path(path: str):
    try:
        p = str(path)
        for name, res in (st.session_state.get("results") or {}).items():
            for m in (res.get("models") or []):
                if m.get("file") == p: return name
    except Exception:
        pass
    return None

# ---------- Sidebar ----------
with st.sidebar:
    st.markdown("## Navigation")
    pages = ["🏠 Home", "📊 Predictions", "📈 Results", "👁️ 3D Visualization", "⚙️ Settings"]
    st.radio("Page", pages, index=pages.index(st.session_state.get("nav_page", "🏠 Home")), key="nav_page")

    st.markdown("## View mode")
    modes = ["pLDDT (structures)", "Identity (alignments)"]
    st.radio("Mode", modes, index=(0 if (st.session_state.get("view_mode", "pLDDT (structures)")).startswith("pLDDT") else 1), key="view_mode")

# ---------- AFDB-first helper ----------
def smart_afdb_fetch(name: str, seq: str | None = None, timeout: int = 10):
    import urllib.request, json, re as _re
    if not name: return None
    candidates = []
    toks = _re.split(r"[|\s,;/]+", str(name))
    for t in toks:
        if _is_uniprot_acc(t): candidates.append(t)
    if not candidates and _re.search(r"\b([A-Z0-9]{6,10})(?:-\d+)?\b", name):
        candidates.append(_re.search(r"\b([A-Z0-9]{6,10}(?:-\d+)?)\b", name).group(1))
    for acc in candidates:
        try_list = [acc]
        if "-" in acc: try_list.append(acc.split("-")[0])
        for acc_try in try_list:
            try:
                url = f"https://alphafold.ebi.ac.uk/api/prediction/{acc_try}"
                with urllib.request.urlopen(url, timeout=timeout) as r:
                    j = json.loads(r.read().decode("utf-8","ignore"))
                if isinstance(j, list) and j:
                    obj = j[0]
                    return {
                        "acc": obj.get("uniprotAccession") or acc_try,
                        "pdb_url": obj.get("pdbUrl"),
                        "cif_url": obj.get("cifUrl"),
                        "bcif_url": obj.get("bcifUrl"),
                    }
            except Exception:
                continue
    return None

def auto_guess_uniprot_acc_from_name(seq_name: str):
    if not seq_name: return None
    if _is_uniprot_acc(seq_name): return seq_name
    m = re.search(r"\b([A-Z0-9]{6,10})\b", seq_name, flags=re.I)
    if m and _is_uniprot_acc(m.group(1)): return m.group(1)
    return None

def auto_guess_uniprot_acc_for_sequence(seq_name: str):
    acc = (st.session_state.get("_afdb_used") or {}).get(seq_name)
    if acc: return acc
    acc = auto_guess_uniprot_acc_from_name(seq_name)
    if acc: return acc
    return None

# ---------- PDB/CIF I/O + pLDDT extraction ----------
def cif_to_pdb_text_cif(cif_path: Path, pdb_out: Path) -> bool:
    try:
        parser = MMCIFParser(QUIET=True); stct = parser.get_structure("S", str(cif_path))
        io = PDBIO(); io.set_structure(stct); io.save(str(pdb_out))
        return pdb_out.exists() and pdb_out.stat().st_size > 0
    except Exception:
        return False

def any_cif_to_pdb(cif_or_bcif_path: Path, pdb_out: Path) -> bool:
    try:
        stct = gemmi.read_structure(str(cif_or_bcif_path))
        txt = stct.make_minimal_pdb()
        open(pdb_out, "w").write(txt)
        return pdb_out.exists() and pdb_out.stat().st_size > 0
    except Exception:
        return False

def extract_plddt_from_pdb(p: Path):
    vals = []
    try:
        with open(p, "r") as f:
            for ln in f:
                if not ln.startswith("ATOM") or len(ln) < 66:
                    continue
                try:
                    b = float(ln[60:66].strip())
                    if 0.0 <= b <= 100.0:
                        vals.append(b)
                except Exception:
                    continue
    except Exception:
        pass
    return vals

def extract_plddt_from_cif(p: Path):
    vals = []
    try:
        stct = gemmi.read_structure(str(p))
        for mod in stct:
            for ch in mod:
                for res in ch:
                    for a in res:
                        try:
                            b = float(a.b_iso)
                            if 0.0 <= b <= 100.0:
                                vals.append(b)
                        except Exception:
                            pass
    except Exception:
        try:
            parser = MMCIFParser(QUIET=True); stct = parser.get_structure("S", str(p))
            for a in stct.get_atoms():
                try:
                    b = float(a.get_bfactor())
                    if 0.0 <= b <= 100.0:
                        vals.append(b)
                except Exception:
                    pass
        except Exception:
            pass
    return vals

def list_polymer_chains(path: str):
    try:
        stct = gemmi.read_structure(str(path))
        chains = []
        if len(stct) > 0:
            for ch in stct[0]:
                # robust: mark polymeric if chain has at least one amino acid
                has_aa = any(res.is_amino_acid() for res in ch)
                if has_aa:
                    chains.append((ch.name, sum(1 for res in ch if res.is_amino_acid())))
        chains = sorted(chains, key=lambda x: x[1], reverse=True)
        return [c for c, _ in chains] if chains else []
    except Exception:
        return []

# ---------- pLDDT helpers + panels ----------
def af_detect_fmt_from_text(txt: str, guess: str | None = None) -> str:
    h = (txt or "").lstrip()[:2000]
    has_cif = ("_atom_site." in h) or h.startswith("data_")
    has_pdb = h.startswith(("ATOM","HETATM","MODEL","HEADER","REMARK"))
    if has_cif and not has_pdb:
        return "cif"
    if has_pdb and not has_cif:
        return "pdb"
    return guess or "pdb"

def af_plddt_per_residue_by_chain_from_text(txt: str, fmt: str | None = None, sel_chain: str | None = None):
    from collections import OrderedDict, defaultdict
    per_chain = {}
    fmt = af_detect_fmt_from_text(txt, fmt)
    if fmt == "pdb":
        ch_map = defaultdict(lambda: OrderedDict())
        for ln in (txt or "").splitlines():
            if not ln.startswith("ATOM"):
                continue
            if len(ln) < 66:
                continue
            if ln[12:16].strip() != "CA":
                continue
            chain = ln[21:22].strip() or "A"
            if sel_chain and chain != sel_chain:
                continue
            key = (ln[22:26].strip(), ln[26:27])
            try:
                b = float(ln[60:66].strip())
                if 0.0 <= b <= 100.0 and key not in ch_map[chain]:
                    ch_map[chain][key] = b
            except Exception:
                pass
        for ch, od in ch_map.items():
            vals = list(od.values())
            if vals:
                per_chain[ch] = vals
        return per_chain
    try:
        import tempfile, os
        with tempfile.NamedTemporaryFile("w", suffix=".cif", delete=False) as f:
            f.write(txt or ""); path = f.name
        stct = gemmi.read_structure(path)
        if len(stct) > 0:
            model = stct[0]
            for ch in model:
                # robust chain check
                has_aa = any(res.is_amino_acid() for res in ch)
                if not has_aa:
                    continue
                if sel_chain and ch.name != sel_chain:
                    continue
                vals = []
                for res in ch:
                    if not res.is_amino_acid():
                        continue
                    ca = res.find_atom("CA","")
                    if ca is None:
                        bs = [float(a.b_iso) for a in res if hasattr(a,'b_iso') and 0.0 <= float(getattr(a,'b_iso',0.0)) <= 100.0]
                        if bs:
                            vals.append(sum(bs)/len(bs))
                            continue
                    try:
                        b = float(ca.b_iso)
                        if 0.0 <= b <= 100.0:
                            vals.append(b)
                    except Exception:
                        pass
                if vals:
                    per_chain[ch.name] = vals
        try:
            os.remove(path)
        except Exception:
            pass
    except Exception:
        pass
    return per_chain

def af_bins_counts(vals):
    b = {"Very high (>90)":0, "High (70–90)":0, "Low (50–70)":0, "Very low (<50)":0}
    for v in vals:
        try:
            f = float(v)
        except Exception:
            continue
        if f < 50:
            b["Very low (<50)"] += 1
        elif f < 70:
            b["Low (50–70)"] += 1
        elif f < 90:
            b["High (70–90)"] += 1
        else:
            b["Very high (>90)"] += 1
    return b, sum(b.values())

def af_plddt_legend_html():
    return """
    <div style="border-radius:10px;overflow:hidden;border:1px solid #dce1ea;max-width:360px;">
      <div style="background:#e9f0ff;padding:10px 12px;font-weight:700;color:#163dff;">Model Confidence</div>
      <div style="padding:10px 12px;background:#fff;">
        <div style="display:flex;align-items:center;margin:6px 0;">
          <span style="display:inline-block;width:14px;height:14px;background:#2166f3;border-radius:2px;margin-right:8px;"></span>
          Very high (pLDDT > 90)
        </div>
        <div style="display:flex;align-items:center;margin:6px 0;">
          <span style="display:inline-block;width:14px;height:14px;background:#8fd3ff;border-radius:2px;margin-right:8px;"></span>
          High (70 ≤ pLDDT < 90)
        </div>
        <div style="display:flex;align-items:center;margin:6px 0;">
          <span style="display:inline-block;width:14px;height:14px;background:#ffd35a;border-radius:2px;margin-right:8px;"></span>
          Low (50 ≤ pLDDT < 70)
        </div>
        <div style="display:flex;align-items:center;margin:6px 0;">
          <span style="display:inline-block;width:14px;height:14px;background:#ff8c42;border-radius:2px;margin-right:8px;"></span>
          Very low (pLDDT < 50)
        </div>
      </div>
    </div>
    """
# (suite: figures pLDDT, render, domaines, ranking, runner, A3M, US-align helpers, etc.)
# ===================== PAGES =====================
''')
# ------------- FIN PARTIE 1/2 -------------
APP.append(r'''
# ---------- pLDDT figures / panels ----------
def build_af_plddt_figs_from_text(txt, fmt=None, title="pLDDT", sel_chain=None):
    per_chain = af_plddt_per_residue_by_chain_from_text(txt, fmt, sel_chain=sel_chain)
    vals = [float(v) for arr in per_chain.values() for v in arr if v is not None and 0.0 <= float(v) <= 100.0]
    fig_hist = px.histogram(x=vals, nbins=40, range_x=[0, 100],
                            labels={"x":"pLDDT","y":"Residues"},
                            title=f"pLDDT distribution — {title}")
    fig_hist.update_layout(margin=dict(l=10, r=10, t=40, b=10), bargap=0.02)
    counts, total = af_bins_counts(vals)
    labels_order = ["Very high (>90)","High (70–90)","Low (50–70)","Very low (<50)"]
    colors = {"Very high (>90)":"#2166f3","High (70–90)":"#8fd3ff","Low (50–70)":"#ffd35a","Very low (<50)":"#ff8c42"}
    fig_pie = px.pie(values=[counts[k] for k in labels_order], names=labels_order, title="pLDDT distribution (AF)")
    fig_pie.update_traces(textposition='inside', textinfo='percent+label',
                          marker=dict(colors=[colors[k] for k in labels_order]))
    fig_pie.update_layout(margin=dict(l=10, r=10, t=40, b=10))
    mean_v = float(np.mean(vals)) if vals else 0.0
    fig_gauge = go.Figure(go.Indicator(
        mode="gauge+number", value=mean_v, number={"suffix":" pLDDT","valueformat":".2f"},
        gauge={"axis":{"range":[0,100]}, "bar":{"color":"#2166f3"},
               "steps":[{"range":[0,50],"color":"#ff8c42"},
                        {"range":[50,70],"color":"#ffd35a"},
                        {"range":[70,90],"color":"#8fd3ff"},
                        {"range":[90,100],"color":"#2166f3"}],
               "threshold":{"line":{"color":"black","width":2},"thickness":0.75,"value":mean_v}},
        title={"text":"Average pLDDT"}, domain={"x":[0,1],"y":[0,1]}
    ))
    fig_gauge.update_layout(margin=dict(l=10, r=10, t=40, b=10))
    return fig_hist, fig_pie, fig_gauge

def build_af_plddt_extras_from_text(txt, fmt=None, sel_chain=None, title="pLDDT"):
    per_chain = af_plddt_per_residue_by_chain_from_text(txt, fmt, sel_chain=sel_chain)
    vals = [float(v) for arr in per_chain.values() for v in arr if v is not None and 0.0 <= float(v) <= 100.0]
    fig_ecdf = px.ecdf(vals, labels={"value":"pLDDT","ECDF":"Cumulative fraction"}, title=f"ECDF — {title}")
    fig_ecdf.update_layout(margin=dict(l=10, r=10, t=40, b=10))
    counts, _ = af_bins_counts(vals)
    labels_order = ["Very low (<50)","Low (50–70)","High (70–90)","Very high (>90)"]
    colors = {"Very high (>90)":"#2166f3","High (70–90)":"#8fd3ff","Low (50–70)":"#ffd35a","Very low (<50)":"#ff8c42"}
    fig_stack = go.Figure()
    total = sum(counts.values()) or 1
    for lab in labels_order:
        v = counts.get(lab, 0)
        fig_stack.add_trace(go.Bar(
            x=[v], y=["pLDDT"], orientation="h",
            marker=dict(color=colors.get(lab, "#ccc")),
            name=lab,
            hovertemplate=f"{lab}: {v} ({100.0*v/total:.1f}%)<extra></extra>"
        ))
    fig_stack.update_layout(barmode="stack", title=f"Stacked pLDDT — {title}",
                            margin=dict(l=10, r=10, t=40, b=10), showlegend=True)
    return fig_ecdf, fig_stack

def display_plddt_panels(txt, fmt, label, sel_chain, key_prefix):
    if not txt:
        st.info("AF pLDDT: empty source."); return
    if st.checkbox("Show pLDDT legend", value=True, key=f"{key_prefix}_legend"):
        st.markdown(af_plddt_legend_html(), unsafe_allow_html=True)
    figH, figP, figG = build_af_plddt_figs_from_text(txt, fmt, title=str(Path(label).name if label else "current"), sel_chain=sel_chain)
    c1, c2 = st.columns([2,1])
    with c1: st.plotly_chart(figH, use_container_width=True)
    with c2:
        st.plotly_chart(figP, use_container_width=True)
        st.plotly_chart(figG, use_container_width=True)
    figE, figS = build_af_plddt_extras_from_text(txt, fmt, sel_chain=sel_chain, title=str(Path(label).name if label else "current"))
    c3, c4 = st.columns(2)
    with c3: st.plotly_chart(figE, use_container_width=True)
    with c4: st.plotly_chart(figS, use_container_width=True)

# ---------- Keep first model (PDB only) ----------
def keep_first_model_text(fmt: str, txt: str) -> str:
    # PDB: keep only the first MODEL block (do not truncate CIF)
    if fmt == "pdb":
        import re as _re
        if _re.search(r'^\s*MODEL', txt, flags=_re.M):
            m = _re.search(r'^\s*MODEL[^\n]*\n(.*?)(?:\nENDMDL|\Z)', txt, flags=_re.S|_re.M)
            if m:
                body = m.group(1)
                lines = [ln for ln in body.splitlines() if ln.startswith(("ATOM","HETATM","ANISOU","TER"))]
                return "\n".join(lines) + "\n"
        return txt
    return txt

# ---------- CIF label_seq_id → auth_seq_id mapping (for domain overlays) ----------
def build_label_to_auth_map_from_text(txt: str, chain: str | None = None) -> dict[int,int]:
    try:
        from gemmi import cif as cifmod
        doc = cifmod.read_string(txt)
        block = doc.sole_block()
        tbl = block.find_mmcif_category('_atom_site')
        if not tbl:
            return {}
        lab = tbl.get('_atom_site.label_seq_id') or []
        aut = tbl.get('_atom_site.auth_seq_id') or []
        lab_asym = tbl.get('_atom_site.label_asym_id') or []
        aut_asym = tbl.get('_atom_site.auth_asym_id') or []
        use_chain = chain is not None
        counts = {}
        n = min(len(lab), len(aut), len(lab_asym) if lab_asym else len(aut), len(aut_asym) if aut_asym else len(aut))
        for i in range(n):
            chain_id = (aut_asym[i] if aut_asym else (lab_asym[i] if lab_asym else None))
            if use_chain and chain_id != chain:
                continue
            try:
                lbl = int(lab[i]); aval = aut[i]
                if not aval or aval in {'.','?'}: continue
                aint = int(aval)
            except Exception:
                continue
            counts[(lbl, aint)] = counts.get((lbl, aint), 0) + 1
        mapping = {}
        for lbl in sorted({k[0] for k in counts}):
            best_a, best_c = None, -1
            for (l, a), c in counts.items():
                if l == lbl and c > best_c:
                    best_c = c; best_a = a
            if best_a is not None:
                mapping[lbl] = best_a
        return mapping
    except Exception:
        return {}

# ---------- Robust pLDDT coloring (AlphaFold 4-color & Special blue/orange) ----------
def render_py3d_html(txt: str, fmt: str, style: str, scheme: str, mono_only=False, sel_chain=None, dark_bg: bool=False):
    """
    Robust coloring without JS colorfunc:
    - AlphaFold (4-color): <50 orange, 50–70 yellow, 70–90 light-blue, >90 dark-blue
    - Special (blue/orange): <70 orange, 70–90 light-blue, >90 dark-blue
    Falls back to py3Dmol colorschemes for pLDDT/Spectrum/Chain.
    """
    import py3Dmol, gemmi, tempfile, os
    viewer = py3Dmol.view(width=1000, height=650)
    viewer.addModel(txt, "cif" if fmt=="cif" else "pdb")
    viewer.setBackgroundColor("black" if dark_bg else "white")
    sel_global = {"chain": sel_chain} if (mono_only and sel_chain) else {}

    def _build_classes(_txt: str, _fmt: str, only_chain: str | None):
        out = {"lt50":{}, "mid50_70":{}, "mid70_90":{}, "ge90":{}, "lt70":{}}
        suffix = ".cif" if _fmt == "cif" else ".pdb"
        fp = tempfile.NamedTemporaryFile("w", suffix=suffix, delete=False)
        fp.write(_txt or ""); fp.flush(); fp.close()
        try:
            st = gemmi.read_structure(fp.name)
            if len(st) > 0:
                model = st[0]
                mapping = None
                try:
                    if _fmt == "cif" and only_chain:
                        mapping = build_label_to_auth_map_from_text(_txt, chain=only_chain)
                except Exception:
                    mapping = None
                for ch in model:
                    has_aa = any(res.is_amino_acid() for res in ch)
                    if not has_aa:
                        continue
                    chname = ch.name
                    if only_chain and chname != only_chain:
                        continue
                    acc = {"lt50":[], "mid50_70":[], "mid70_90":[], "ge90":[], "lt70":[]}
                    for res in ch:
                        if not res.is_amino_acid():
                            continue
                        v = None
                        ca = res.find_atom("CA","")
                        if ca:
                            try: v = float(ca.b_iso)
                            except Exception: v = None
                        if v is None:
                            vals = []
                            for a in res:
                                try:
                                    b = float(a.b_iso)
                                    if 0.0 <= b <= 100.0: vals.append(b)
                                except Exception: pass
                            v = (sum(vals)/len(vals)) if vals else None
                        resi = None
                        try: resi = int(res.seqid.num)
                        except Exception: resi = None
                        if _fmt == "cif" and mapping:
                            try:
                                lbl = int(res.seqid.num)
                                resi = mapping.get(lbl, resi)
                            except Exception: pass
                        if v is None or resi is None:
                            continue
                        if v < 50.0:
                            acc["lt50"].append(int(resi)); acc["lt70"].append(int(resi))
                        elif v < 70.0:
                            acc["mid50_70"].append(int(resi)); acc["lt70"].append(int(resi))
                        elif v < 90.0:
                            acc["mid70_90"].append(int(resi))
                        else:
                            acc["ge90"].append(int(resi))
                    for k, lst in acc.items():
                        if lst: out[k].setdefault(chname, []).extend(sorted(set(lst)))
        finally:
            try: os.remove(fp.name)
            except Exception: pass
        return out

    _fmt = fmt.lower().strip()
    if _fmt not in ("pdb","cif"):
        _fmt = "cif" if ("_atom_site." in (txt or "")) else "pdb"
    classes = _build_classes(txt, _fmt, sel_chain if mono_only else None)

    AF_COLORS = {"lt50": "#FF8C42", "mid50_70": "#FFD35A", "mid70_90": "#8FD3FF", "ge90": "#2166F3"}
    BO_COLORS = {"lt70": "#FF7F0E", "ge70_lo": "#4CA6FF", "ge90": "#1F57F7"}

    rep = (style or "Cartoon").strip().lower()
    rep = rep if rep in ("cartoon","stick","sphere","line","surface") else "cartoon"

    def _apply(sel, color):
        if rep == "cartoon": viewer.setStyle(sel, {"cartoon": {"color": color}})
        elif rep == "stick": viewer.setStyle(sel, {"stick": {"color": color, "radius": 0.3}})
        elif rep == "sphere": viewer.setStyle(sel, {"sphere": {"color": color, "radius": 1.0}})
        elif rep == "line": viewer.setStyle(sel, {"line": {"color": color}})
        elif rep == "surface": viewer.setStyle(sel, {"surface": {"opacity": 0.85, "color": color}})

    if scheme == "AlphaFold (4-color)":
        for cls, color in (("lt50", AF_COLORS["lt50"]),
                           ("mid50_70", AF_COLORS["mid50_70"]),
                           ("mid70_90", AF_COLORS["mid70_90"]),
                           ("ge90", AF_COLORS["ge90"])):
            for chn, reslist in (classes.get(cls, {}) or {}).items():
                if not reslist: continue
                sel = {"resi": reslist}; sel.update(sel_global or {})
                if not mono_only: sel["chain"] = chn
                _apply(sel, color)
    elif scheme == "Special (blue/orange)":
        for chn, reslist in (classes.get("lt70", {}) or {}).items():
            if not reslist: continue
            sel = {"resi": reslist}; sel.update(sel_global or {})
            if not mono_only: sel["chain"] = chn
            _apply(sel, BO_COLORS["lt70"])
        for chn, reslist in (classes.get("ge90", {}) or {}).items():
            if not reslist: continue
            sel = {"resi": reslist}; sel.update(sel_global or {})
            if not mono_only: sel["chain"] = chn
            _apply(sel, BO_COLORS["ge90"])
        for chn, reslist in (classes.get("mid70_90", {}) or {}).items():
            if not reslist: continue
            sel = {"resi": reslist}; sel.update(sel_global or {})
            if not mono_only: sel["chain"] = chn
            _apply(sel, BO_COLORS["ge70_lo"])
    else:
        # fallback builtins
        if scheme == "pLDDT (B-factor)":
            cs = {"prop":"b","gradient":"roygb","min":0,"max":100}; sty = {"colorscheme": cs}
        elif scheme == "Spectrum":
            sty = {"colorscheme": "spectrum"}
        elif scheme == "Chain":
            sty = {"colorscheme": "chain"}
        else:
            cs = {"prop":"b","gradient":"roygb","min":0,"max":100}; sty = {"colorscheme": cs}
        if rep == "cartoon": viewer.setStyle(sel_global, {"cartoon": sty})
        elif rep == "stick": st = dict(sty); st["radius"] = 0.3; viewer.setStyle(sel_global, {"stick": st})
        elif rep == "sphere": st = dict(sty); st["radius"] = 1.0; viewer.setStyle(sel_global, {"sphere": st})
        elif rep == "line": viewer.setStyle(sel_global, {"line": sty})
        elif rep == "surface": viewer.setStyle(sel_global, {"surface": {"opacity": 0.85}})

    viewer.zoomTo(); viewer.render()
    return viewer._make_html()

# ---------- Domain overlay viewer ----------
def render_domains_py3d_html(txt: str, fmt: str, segs: list, style: str = "Cartoon", chain: str | None = None, use_label_seq_id: bool = False):
    viewer = py3Dmol.view(width=1000, height=650)
    viewer.addModel(txt, "cif" if fmt=="cif" else "pdb")
    viewer.setStyle({}, {"cartoon": {"color": "#DDDDDD"}})
    mapping = None
    if fmt == "cif" and use_label_seq_id and chain:
        mapping = build_label_to_auth_map_from_text(txt, chain=chain)
    for i, s in enumerate(segs):
        color = _domain_color(i)
        start_i = int(s.get("start", 1)); end_i = int(s.get("end", start_i))
        if mapping:
            resi_list = [mapping.get(k) for k in range(start_i, end_i+1)]
            resi_list = [int(x) for x in resi_list if x is not None]
            sel = {"resi": resi_list} if resi_list else {"resi": list(range(start_i, end_i+1))}
        else:
            sel = {"resi": list(range(start_i, end_i+1))}
        if chain: sel["chain"] = chain
        if style == "Cartoon": viewer.setStyle(sel, {"cartoon": {"color": color}})
        elif style == "Stick": viewer.setStyle(sel, {"stick": {"color": color, "radius": 0.3}})
        elif style == "Sphere": viewer.setStyle(sel, {"sphere": {"color": color, "radius": 1.0}})
        elif style == "Line": viewer.setStyle(sel, {"line": {"color": color}})
        else: viewer.setStyle(sel, {"cartoon": {"color": color}})
    viewer.setBackgroundColor("white"); viewer.zoomTo()
    return viewer._make_html()
''')
APP.append(r'''
# ---------- Ranking + models ----------
def parse_ranking(job_dir: Path):
    files = list(Path(job_dir).rglob("ranking_debug.json"))
    if not files:
        files = list(Path(job_dir).rglob("ranking*.json"))
    if not files:
        return {}
    try:
        d = json.load(open(files[0], "r"))
        out = {}
        if "order" in d and isinstance(d["order"], list):
            out["order"] = d["order"]
        for k in ("ranking_confidence","plddts","iptms","ptms"):
            if k in d and isinstance(d[k], dict):
                out[k] = d[k]
        return out
    except Exception:
        return {}

def safe_avg_plddt_from_path(path: Path, fmt: str):
    vals = extract_plddt_from_pdb(path) if fmt=="pdb" else extract_plddt_from_cif(path)
    arr = []
    for v in vals:
        try:
            vv = float(v)
            if vv < 0: vv = 0.0
            if vv > 100: vv = 100.0
            arr.append(vv)
        except Exception:
            pass
    return float(np.mean(arr)) if arr else None

def harvest_models(job_dir: Path):
    rank = parse_ranking(job_dir)
    pdbs = sorted(Path(job_dir).rglob("*.pdb"), key=lambda x: x.stat().st_mtime)
    cifs = sorted(list(Path(job_dir).rglob("*.cif")) + list(Path(job_dir).rglob("*.bcif")), key=lambda x: x.stat().st_mtime)
    converted = []
    if not pdbs and cifs:
        for c in cifs:
            out = c.with_suffix("").with_name(c.stem + "_converted.pdb")
            try:
                if any_cif_to_pdb(c, out) or (c.suffix.lower() == ".cif" and cif_to_pdb_text_cif(c, out)):
                    converted.append(out)
            except Exception:
                pass
        pdbs = sorted(converted, key=lambda x: x.stat().st_mtime)

    def model_id_from_name(p: Path):
        name = p.name
        m = re.search(r"(model_\d+)", name)
        if m: return m.group(1)
        m = re.search(r"ranked_(\d+)", name)
        if m:
            try: return f"model_{int(m.group(1))+1}"
            except Exception: pass
        m = re.search(r"unrelaxed_rank_\d+_model_(\d+)", name)
        if m: return f"model_{m.group(1)}"
        m = re.search(r"model_(\d+)_", name)
        if m: return f"model_{m.group(1)}"
        return None

    def guess_rank_from_name(name: str):
        m = re.search(r"ranked_(\d+)", name)
        if m:
            try: return int(m.group(1))+1
            except Exception: return None
        m = re.search(r"rank_(\d+)", name)
        if m:
            try: return int(m.group(1))
            except Exception: return None
        return None

    all_entries = [("pdb", p) for p in pdbs] + [("cif", c) for c in cifs]
    order = rank.get("order", []) if isinstance(rank, dict) else []

    rows = []
    for fmt, path in all_entries:
        mid = model_id_from_name(path)
        avg = safe_avg_plddt_from_path(path, fmt)
        rc  = (rank.get("ranking_confidence", {}) or {}).get(mid) if mid else None
        ptm = (rank.get("ptms", {}) or {}).get(mid) if mid else None
        iptm= (rank.get("iptms", {}) or {}).get(mid) if mid else None
        if mid and order and mid in order:
            rpos = order.index(mid) + 1
        else:
            rpos = guess_rank_from_name(path.name)
        rows.append({
            "model_id": mid or "-",
            "file": str(path),
            "fmt": fmt,
            "avg_plddt": avg,
            "rank": rpos,
            "ranking_conf": rc,
            "ptm": ptm,
            "iptm": iptm
        })
    rows = sorted(rows, key=lambda x: (x["rank"] if x["rank"] is not None else 9999, -(x["avg_plddt"] or 0)))
    return rows, rank

def analyze_results(job_dir: Path):
    rows, rank = harvest_models(job_dir)
    res = {"status":"error","message":"No PDB/CIF","models":rows,"ranking":rank,"coverage_png":None,"pae_json":None}
    if rows:
        res["status"] = "success"; res["message"] = ""
    covs = list(Path(job_dir).rglob("*coverage*.png"))
    if covs:
        res["coverage_png"] = str(sorted(covs, key=lambda x: x.stat().st_mtime)[-1])
    pae = list(Path(job_dir).rglob("*pae*.json"))
    if pae:
        res["pae_json"] = str(sorted(pae, key=lambda x: x.stat().st_mtime)[-1])
    return res

# ---------- PAE heatmap ----------
def render_pae_heatmap(pae_json_path: str, title="PAE Heatmap"):
    try:
        with open(pae_json_path, "r") as f:
            j = json.load(f)
        mat = j.get("pae") or j.get("predicted_aligned_error")
        if mat is None:
            return None
        vmax = max(30.0, float(np.nanmax(np.array(mat))))
        fig = px.imshow(mat, color_continuous_scale="Viridis", origin="lower",
                        labels=dict(color="PAE (Å)"), title=title, zmin=0, zmax=vmax, aspect="auto")
        fig.update_layout(margin=dict(l=10,r=10,t=40,b=10))
        return fig
    except Exception:
        return None

# ---------- ColabFold runner + utils (speed-optimized) ----------
def estimate_msa_timeout(seq_len: int, msa_mode: str, params: dict | None = None) -> int:
    seq_len = int(seq_len or 300); params = params or {}
    if msa_mode == "mmseqs2_uniref_env": base = 3000
    elif msa_mode == "mmseqs2_uniref":   base = 1800
    else:                                 base = 600
    models = max(1, int(params.get("num_models", 1)))
    recycles = max(1, int(params.get("num_recycles", 3)))
    multimer = 1.5 if "multimer" in str(params.get("model_type","")).lower() else 1.0
    size_factor = min(4.0, max(1.0, seq_len/300.0))
    factor = models * max(1.0, recycles/3.0) * multimer * size_factor
    total = int(base * factor)
    return max(600, min(6*3600, total))

def _run(cmd: list[str], timeout_sec: int | None = None, env: dict | None = None, cwd: str | None = None):
    try:
        p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                           env=env, cwd=cwd, timeout=timeout_sec, check=False)
        out = p.stdout or ""
        return (p.returncode == 0), out
    except subprocess.TimeoutExpired as e:
        out = (e.stdout or "") + "\n[timeout]"
        return False, out
    except Exception as e:
        return False, str(e)

def run_colabfold_advanced(fasta_file: str, output_dir: str, params: dict, seq_len: int = 0):
    """
    Wrapper ColabFold batch:
      - --recompile-padding 1
      - --disable-unified-memory when GPU present
      - optional --max-msa cap
      - do not overwrite when reuse_cache=True
    """
    import shutil as _sh, sys, os
    exe = _sh.which("colabfold_batch")
    cmd = [exe] if exe else [sys.executable, "-m", "colabfold.batch"]
    cmd += [fasta_file, output_dir]

    msa_key = str(params.get("msa_strategy","fast")).lower()
    msa_map = {"full": "mmseqs2_uniref_env", "fast": "mmseqs2_uniref", "minimal": "single_sequence"}
    msa_mode = msa_map.get(msa_key, "mmseqs2_uniref")

    if params.get("model_type"): cmd += ["--model-type", str(params["model_type"])]
    cmd += ["--msa-mode", msa_mode]
    if params.get("pair_mode"):  cmd += ["--pair-mode", str(params["pair_mode"])]
    cmd += ["--num-models", str(int(params.get("num_models", 1)))]
    cmd += ["--num-recycle", str(int(params.get("num_recycles", 3)))]
    cmd += ["--rank", "auto"]
    jobname = params.get("jobname_prefix") or "job"
    cmd += ["--jobname", jobname]

    if params.get("use_templates"): cmd += ["--templates"]
    if params.get("use_amber"):     cmd += ["--amber"]
    if params.get("stop_at_score") is not None:
        try: cmd += ["--stop-at-score", f"{float(params['stop_at_score']):.3f}"]
        except Exception: pass

    if params.get("_max_msa"): cmd += ["--max-msa", str(params["_max_msa"]), "--disable-cluster-profile"]
    cmd += ["--recompile-padding", "1"]
    if _sh.which("nvidia-smi"): cmd += ["--disable-unified-memory"]
    if not params.get("reuse_cache", True): cmd += ["--overwrite-existing-results"]

    env = os.environ.copy()
    env.setdefault("TF_CPP_MIN_LOG_LEVEL","3")
    env.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE","false")
    env.setdefault("TF_FORCE_GPU_ALLOW_GROWTH","true")

    timeout = estimate_msa_timeout(seq_len, msa_mode, params)
    ok, log = _run(cmd, timeout_sec=timeout, env=env)
    return ok, log

def validate_prediction(metrics: dict) -> list[str]:
    warns = []
    try:
        plddt = metrics.get("avg_plddt")
        if plddt is not None and plddt < 70:
            warns.append(f"Low mean pLDDT ({plddt:.1f}): caution advised.")
        iptm = metrics.get("iptm")
        if iptm is not None and iptm < 0.4:
            warns.append(f"Low ipTM ({iptm:.2f}): interfaces likely uncertain.")
        ptm = metrics.get("ptm")
        if ptm is not None and ptm < 0.6:
            warns.append(f"Low pTM ({ptm:.2f}): global topology possibly uncertain.")
    except Exception:
        pass
    return warns

# ---------- Identity mode: robust A3M pipeline ----------
def _find_a3m_files(job_dir: Path):
    try:
        jd = Path(job_dir)
        if not jd.exists(): return []
        cand = []
        for pat in ("*.a3m", "*.a3m.gz"): cand.extend(jd.rglob(pat))
        def score(p: Path):
            n = p.name.lower(); s = 0
            if "uniref" in n: s += 100
            if "bfd" in n: s += 50
            if "paired" in n: s += 10
            if "monomer" in n: s += 5
            try: t = p.stat().st_mtime
            except Exception: t = 0
            return (s, t)
        return sorted(set(cand), key=score, reverse=True)
    except Exception:
        return []

def _read_a3m_text(a3m_path: Path) -> str:
    try:
        if str(a3m_path).lower().endswith(".gz"):
            import gzip
            with gzip.open(a3m_path, "rt", encoding="utf-8", errors="ignore") as f:
                return f.read()
        else:
            return open(a3m_path, "r", encoding="utf-8", errors="ignore").read()
    except Exception:
        try:
            if str(a3m_path).lower().endswith(".gz"):
                import gzip
                return gzip.decompress(open(a3m_path, "rb").read()).decode("utf-8", "ignore")
            return open(a3m_path, "rb").read().decode("utf-8", "ignore")
        except Exception:
            return ""

def _parse_a3m_file(a3m_path: Path):
    if not a3m_path or not a3m_path.exists(): return None
    txt = _read_a3m_text(a3m_path)
    lines = txt.splitlines()
    entries = []; head = None; seq = []
    for ln in lines:
        if ln.startswith(">"):
            if head is not None:
                entries.append((head,"".join(seq))); seq = []
            head = ln[1:].strip()
        else:
            seq.append(ln.strip())
    if head is not None: entries.append((head,"".join(seq)))
    if not entries: return None
    qh, q_aln = entries[0]
    hits = []
    for hdr, aln in entries[1:]:
        toks = hdr.split()
        acc = toks[0] if toks else hdr
        meta = {"score": None, "id_frac": None, "id_pct": None, "evalue": None,
                "q_start":None,"q_end":None,"q_len":None,"t_start":None,"t_end":None,"t_len":None}
        if len(toks) >= 4:
            def _f(x):
                try: return float(str(x).replace("E","e"))
                except Exception: return None
            def _i(x):
                try: return int(x)
                except Exception: return None
            meta["score"] = _f(toks[1])
            try:
                v = float(toks[2].rstrip("%"))
                meta["id_frac"] = (v/100.0 if v>1.0 else v)
                meta["id_pct"]  = (v if v>1.0 else v*100.0)
            except Exception:
                pass
            meta["evalue"] = _f(toks[3])
            if len(toks) >= 10:
                meta.update({"q_start":_i(toks[4]),"q_end":_i(toks[5]),"q_len":_i(toks[6]),
                             "t_start":_i(toks[7]),"t_end":_i(toks[8]),"t_len":_i(toks[9])})
        hits.append({"acc":acc, "aln":aln, "hdr_meta":meta})
    return {"query_header": qh, "query_aln": q_aln, "hits": hits}

def _reconstruct_a3m_pair(query_aln: str, target_aln: str):
    q_line=[]; t_line=[]; iq=0
    for ch in target_aln:
        if ch.islower():
            t_line.append(ch); q_line.append("-")
        else:
            t_line.append(ch)
            q_line.append(query_aln[iq] if iq < len(query_aln) else "-"); iq+=1
    return "".join(q_line), "".join(t_line)

def _reconstruct_a3m_pair_robust(query_aln: str, target_aln: str):
    qi, ti = 0, 0
    q_line, t_line = [], []
    Q, T = query_aln, target_aln
    while qi < len(Q) or ti < len(T):
        qch = Q[qi] if qi < len(Q) else None
        tch = T[ti] if ti < len(T) else None
        if qch is not None and qch.islower():
            q_line.append(qch); t_line.append("-"); qi += 1; continue
        if tch is not None and tch.islower():
            t_line.append(tch); q_line.append("-"); ti += 1; continue
        if qch is not None:
            q_line.append(qch)
            if tch is not None:
                t_line.append(tch); ti += 1
            else:
                t_line.append("-")
            qi += 1
        elif tch is not None:
            t_line.append(tch); q_line.append("-"); ti += 1
    return "".join(q_line), "".join(t_line)

def _identity_and_cov(q_line, t_line, q_len):
    a = 0; n = 0; cov_count = 0
    for q, t in zip(q_line, t_line):
        if q != "-":
            cov_count += 1
            if t != "-":
                n += 1; a += (q.upper() == t.upper())
    ident = (100.0*a/n) if n > 0 else None
    cov = round(100.0*cov_count/max(1, q_len), 1) if q_len else None
    return ident, cov, n

def _build_auto_from_a3m(name: str, job_dir: Path):
    files = _find_a3m_files(job_dir)
    if not files: return None
    parsed = None
    for f in files[:6]:
        parsed = _parse_a3m_file(f)
        if parsed: break
    if not parsed: return None
    q_aln = parsed["query_aln"]
    q_seq = re.sub(r"[^A-Za-z]", "", q_aln).upper()
    q_len = len(q_seq)
    enriched = []
    for h in parsed["hits"]:
        try:
            q_line, t_line = _reconstruct_a3m_pair_robust(q_aln, h["aln"])
            if len(q_line) != len(t_line):
                raise ValueError("A3M lengths mismatch")
        except Exception:
            q_line, t_line = _reconstruct_a3m_pair(q_aln, h["aln"])
        meta = h.get("hdr_meta") or {}
        ident_calc, cov_calc, core = _identity_and_cov(q_line, t_line, q_len)
        ident_pct = meta.get("id_pct") if meta.get("id_pct") is not None else ((meta.get("id_frac") or 0.0)*100.0 if meta.get("id_frac") is not None else ident_calc)
        enriched.append({
            "acc": h["acc"], "q_line": q_line, "t_line": t_line,
            "Identity_%": ident_pct, "Score": meta.get("score"), "Evalue": meta.get("evalue"),
            "Cov_query_%": cov_calc, "Aligned_core": core
        })
    def _val(x,k,default):
        v = x.get(k)
        if v is None or (isinstance(v,float) and not np.isfinite(v)): return default
        return v
    enriched = sorted(enriched,
                      key=lambda x: (_val(x,"Identity_%",-1.0),
                                     _val(x,"Score",-1.0),
                                     -_val(x,"Evalue", float("+inf")),
                                     _val(x,"Cov_query_%",-1.0),
                                     _val(x,"Aligned_core",-1.0)),
                      reverse=True)
    return {"source":"auto-a3m","name":f"{name} (AUTO)","query_header":parsed["query_header"],"query_seq":q_seq,"query_len":q_len,"hits":enriched}

def parse_custom_text_results(txt: str):
    import re as _re
    lines = (txt or "").strip().splitlines()
    if not lines: return None
    entries = []; head = None; seq = []
    for ln in lines:
        if ln.startswith(">"):
            if head is not None:
                entries.append((head, "".join(seq))); seq = []
            head = ln[1:].strip()
        else:
            seq.append(ln.strip())
    if head is not None: entries.append((head, "".join(seq)))
    if len(entries) < 2: return None
    qh, qseq = entries[0]
    hits = []
    for h, s in entries[1:]:
        acc = _re.split(r"\s+", h)[0]
        hits.append({"acc": acc, "aln": s, "hdr_meta": {}})
    q_seq = _re.sub(r"[^A-Za-z]", "", qseq).upper()
    return {"name": f"Manual {qh}", "query_header": qh, "query_seq": q_seq, "query_len": len(q_seq), "hits": hits}

# ---------- US-align helpers (TM-align compatible) ----------
def _ensure_usalign_binary() -> str | None:
    import os, stat, urllib.request
    target = "/usr/local/bin/USalign"
    if os.path.exists(target) and os.access(target, os.X_OK):
        return target
    urls = [
        "https://zhanggroup.org/US-align/download/USalign_linux",
        "https://zhanggroup.org/US-align/download/USalign",
        "http://zhanglab.ccmb.med.umich.edu/US-align/USalign"
    ]
    for url in urls:
        try:
            urllib.request.urlretrieve(url, target)
            os.chmod(target, stat.S_IRUSR|stat.S_IWUSR|stat.S_IXUSR|stat.S_IRGRP|stat.S_IXGRP|stat.S_IROTH|stat.S_IXOTH)
            return target
        except Exception:
            continue
    return None

def _run_usalign(pdb_ref: str, pdb_mob: str, out_dir: str):
    import subprocess as _sp, re as _re, os as _os
    us = _ensure_usalign_binary()
    if not us:
        return False, "USalign not available", None, None, None, None
    out_prefix = _os.path.join(out_dir, "usalign_out")
    cmd = [us, pdb_mob, pdb_ref, "-o", out_prefix]
    try:
        p = _sp.run(cmd, stdout=_sp.PIPE, stderr=_sp.STDOUT, text=True, check=False)
        out = p.stdout or ""
        m_tm = _re.search(r"TM-score\s*=\s*([0-9\.]+)", out)
        m_rmsd = _re.search(r"RMSD\s*=\s*([0-9\.]+)", out)
        m_lali = _re.search(r"Aligned length\s*=\s*([0-9]+)", out)
        tm = float(m_tm.group(1)) if m_tm else None
        rmsd = float(m_rmsd.group(1)) if m_rmsd else None
        lali = int(m_lali.group(1)) if m_lali else None
        sup_pdb = out_prefix + "_sup.pdb"
        ok_sup = _os.path.exists(sup_pdb) and _os.path.getsize(sup_pdb) > 0
        return (p.returncode == 0 and ok_sup), out, tm, rmsd, lali, (sup_pdb if ok_sup else None)
    except Exception as e:
        return False, str(e), None, None, None, None
''')
APP.append(r'''
# ===================== PAGES =====================
current_page = ss.get("nav_page", "🏠 Home")

# ---------- UI: Home ----------
if current_page == "🏠 Home":
    st.markdown('<div class="feature-card"><b>🎯 About</b><br/>AlphaFold-faithful ColabFold predictions (rich MSA, paired multimer, adapted recycles) + pLDDT/Identity + AFDB + InterPro/UniProt domains. Built to be organism-agnostic (bacteria, parasites, viruses, mammals, plants, etc.).</div>', unsafe_allow_html=True)
    st.markdown('<div class="info-card"><b>📋 FASTA format</b><pre>>Name_or_Accession\nSEQUENCE\n\n>Complex_AB\nSEQ_A:SEQ_B</pre></div>', unsafe_allow_html=True)

# ---------- UI: Predictions ----------
elif current_page == "📊 Predictions":
    st.markdown('<div class="sub-header">🔬 Prediction Configuration</div>', unsafe_allow_html=True)
    fa = st.text_area("FASTA (1 block per protein/complex):", key="fasta_text", height=220, placeholder=">P00533\nSEQUENCE\n\n>MyProtein\nSEQUENCE")
    valid, warns = ([], [])
    if fa.strip():
        def parse_fasta_complete(text: str):
            seqs = []; head = None; buf = []
            for ln in (text or "").splitlines():
                ln = ln.strip()
                if not ln:
                    if head and buf: seqs.append((head, "".join(buf))); head = None; buf = []
                    continue
                if ln.startswith(">"):
                    if head and buf: seqs.append((head, "".join(buf)))
                    head = ln[1:].strip(); buf = []
                else:
                    if head is None and not buf: head = ln
                    else: buf.append(ln)
            if head and buf: seqs.append((head, "".join(buf)))
            return seqs
        seqs = parse_fasta_complete(fa)
        tmp = []; warns = []
        for name, seq in seqs:
            s = clean_sequence_advanced(seq)
            if not s:
                warns.append(f"❌ {name}: empty sequence after cleaning"); continue
            tmp.append((name or 'sequence', s))
        valid = tmp
        if valid: st.success(f"{len(valid)} valid sequence(s).")
        for w in warns: st.warning(w)

    tab1, tab2 = st.tabs(["🎯 Base", "🔧 Advanced"])
    with tab1:
        c1, c2, c3 = st.columns(3)
        with c1:
            quality = st.selectbox("Quality profile", ["Classic","Fast","High precision"], index=0)
            msa_strategy = st.selectbox("MSA Strategy", ["Complete (UniRef+Environ.)","Fast (UniRef)","Minimal (single)"], index=1)
        with c2:
            num_models = st.slider("Number of models", 1, 5, 3)
            base_recycles = st.slider("Recycle cycles (min)", 1, 20, 6)
        with c3:
            use_templates = st.checkbox("Templates (PDB)", True)
            use_amber = st.checkbox("Relax AMBER", False)
    with tab2:
        c1, c2 = st.columns(2)
        with c1:
            model_type_ui = st.selectbox("Model type (auto recommended)", ["auto","alphafold2_multimer_v3","alphafold2_ptm"], 0)
            pair_mode_ui = st.selectbox("MSA pairing", ["auto","paired","unpaired","unpaired_paired"], 0)
        with c2:
            stop_at_score = st.number_input("Stop at mean pLDDT (0 = off)", 0.0, 100.0, 0.0, 1.0)
            purge_old = st.checkbox("🧹 Purge old results (optional)", False)

    strict_mono = st.checkbox("🧪 Strict monomer mode (disables pair-mode)", value=False)
    no_homologs = st.checkbox("🔬 Protein without known homologs (disables templates)", value=False)

    fast_afdb_skip = st.checkbox("⚡ Fast for known proteins (use AFDB if available, skip ColabFold)", value=False,
                                 help="Leave unchecked if you want to generate alignments (.a3m)")
    turbo_quick = st.checkbox("🚀 Turbo quick mode if no AFDB (single_sequence • 1 model • 2 recycles)", value=False)

    auto_speed_opt = st.checkbox("⚙️ Auto speed optimize when Strict monomer or No homologs", value=True,
                                 help="Forces 1 model, ≤3 recycles, unpaired, templates OFF, stop_at_score=75, MSA cap 64:64")
    reuse_cache = st.checkbox("♻️ Reuse cached results (MSA/outputs)", value=True)
    msa_cap = st.selectbox("Limit MSA size", ["Default","128:128","96:96","64:64"], index=3,
                           help="Smaller is faster. 64:64 is a good speed/quality balance.")

    msa_map = {"Complete (UniRef+Environ.)":"full","Fast (UniRef)":"fast","Minimal (single)":"minimal"}
    msa_strategy_key = msa_map.get(msa_strategy, "fast")

    if st.button("🚀 Launch Predictions", type="primary", use_container_width=True):
        if not valid:
            st.error("No valid sequences.")
        else:
            if purge_old:
                shutil.rmtree(RESULTS_DIR, ignore_errors=True)
                RESULTS_DIR.mkdir(parents=True, exist_ok=True)
            run_root = new_run_root()
            ss.results = {}; ss.job_dirs = {}; ss.fasta_paths = {}
            prog = st.progress(0); info = st.empty(); results = {}

            for i, (name, seq) in enumerate(valid):
                safe = make_safe_basename(name, seq, maxlen=80)
                fa_path = str(Path("/tmp")/f"{safe}.fasta"); open(fa_path,"w").write(f">{safe}\n{seq}\n")

                is_complex = detect_complex(seq)
                seq_len = seq_length_total(seq)

                # AFDB-first (skip CF if found)
                if fast_afdb_skip:
                    try:
                        fetch = smart_afdb_fetch(name, seq)
                        if fetch:
                            url = fetch.get("pdb_url") or fetch.get("cif_url") or fetch.get("bcif_url")
                            if url:
                                out_dir = (run_root / f"{safe}_{hashlib.sha1(seq.encode()).hexdigest()[:8]}")
                                out_dir.mkdir(parents=True, exist_ok=True)
                                local = out_dir / (safe + "_AFDB" + Path(url).suffix)
                                if not local.exists():
                                    urllib.request.urlretrieve(url, local)
                                (st.session_state["_afdb_used"]).setdefault(name, fetch.get("acc"))
                                if local.suffix.lower() in [".cif",".bcif"]:
                                    outp = local.with_suffix("").with_name(local.stem + "_converted.pdb")
                                    try:
                                        _ = any_cif_to_pdb(local, outp) or (local.suffix.lower()==".cif" and cif_to_pdb_text_cif(local, outp))
                                    except Exception:
                                        pass
                                res = analyze_results(out_dir)
                                if res.get("status") == "success":
                                    ss.job_dirs[name] = str(out_dir); ss.fasta_paths[name] = fa_path; results[name] = res
                                    best = (sorted(res["models"], key=lambda m: (m["rank"] if m["rank"] is not None else 9999, -(m["avg_plddt"] or 0))) or [None])[0]
                                    if best:
                                        st.success(f"⚡ {name}: AFDB used (skipped ColabFold) • {best['model_id']} • pLDDT≈{(best['avg_plddt'] or 0):.1f}")
                                    else:
                                        st.success(f"⚡ {name}: AFDB used (skipped ColabFold) • structure found")
                                    prog.progress((i+1)/len(valid)); continue
                    except Exception as e:
                        st.info(f"AFDB-fast: {e}")

                # ColabFold parameters (quality → recycles)
                if quality == "Fast":
                    rec_rec = 6 if is_complex else 3
                elif quality == "High precision":
                    rec_rec = 20 if is_complex else 12
                else:
                    rec_rec = 12 if is_complex else 6
                num_recycles_eff = 2 if turbo_quick else rec_rec

                if is_complex:
                    model_type = "alphafold2_multimer_v3" if model_type_ui=="auto" else model_type_ui
                    pair_mode  = "paired" if pair_mode_ui=="auto" else pair_mode_ui
                else:
                    model_type = "alphafold2_ptm" if model_type_ui=="auto" else model_type_ui
                    pair_mode  = ("unpaired" if strict_mono else ("unpaired_paired" if pair_mode_ui=="auto" else pair_mode_ui))

                use_templates_eff = False if turbo_quick else (use_templates and (not no_homologs))

                params = {
                    "num_models": (1 if turbo_quick else num_models),
                    "num_recycles": num_recycles_eff,
                    "use_amber": (False if turbo_quick else use_amber),
                    "use_templates": use_templates_eff,
                    "model_type": model_type,
                    "pair_mode": ("paired" if (is_complex and turbo_quick) else ("unpaired" if (not is_complex and turbo_quick) else pair_mode)),
                    "stop_at_score": (70.0 if turbo_quick else (stop_at_score if stop_at_score>0 else None)),
                    "msa_strategy": ("minimal" if turbo_quick else msa_strategy_key),
                    "jobname_prefix": safe,
                    "_quick_minimal": bool(turbo_quick),
                    "reuse_cache": bool(reuse_cache),
                    "_max_msa": (None if msa_cap=="Default" else msa_cap),
                }

                # Auto speed optimize quand monomère/no-homologs sans AFDB-fast/Turbo
                if auto_speed_opt and (strict_mono or no_homologs) and (not fast_afdb_skip) and (not turbo_quick):
                    if params["msa_strategy"] != "minimal":
                        params["msa_strategy"] = "fast"
                    if not params.get("_max_msa"):
                        params["_max_msa"] = "64:64"
                    params["num_models"] = 1
                    params["num_recycles"] = min(params["num_recycles"], 3)
                    params["pair_mode"] = "unpaired"
                    params["use_templates"] = False
                    if params.get("stop_at_score") is None or params.get("stop_at_score") <= 0:
                        params["stop_at_score"] = 75.0

                info.text(f"🔬 {name} — model={params['model_type']} • MSA={params['msa_strategy']} • recycles={params['num_recycles']} • pair={params['pair_mode']} • templates={'ON' if params['use_templates'] else 'OFF'}")

                # Stable cache dir (reuse across runs) or per-run dir
                seq_hash = hashlib.sha1(seq.encode()).hexdigest()[:10]
                key_long = f"{safe}_{seq_hash}_{params['msa_strategy']}_{params['model_type']}_{params['pair_mode']}_{params['num_models']}m_{params['num_recycles']}r_{'tmpl' if params['use_templates'] else 'notmpl'}_{params.get('_max_msa','NA')}"
                key_hash = hashlib.sha1(key_long.encode()).hexdigest()[:10]
                cache_key = (key_long[:150] + "_" + key_hash) if len(key_long) > 190 else key_long
                out_dir = (CACHE_DIR / cache_key) if reuse_cache else (run_root / f"{safe}_{seq_hash}")
                out_dir.mkdir(parents=True, exist_ok=True)

                ok, log = run_colabfold_advanced(fa_path, str(out_dir), params, seq_len=seq_len)
                job_dir = out_dir
                res = analyze_results(job_dir)

                if res.get("status") != "success":
                    st.info("⛑️ No PDB/CIF → trying last‑chance CPU…")
                    def last_chance_minirun(fasta_file: str, output_dir: str, job_prefix: str, is_multimer: bool):
                        exe = shutil.which("colabfold_batch")
                        cmd = [exe or sys.executable, *([] if exe else ["-m","colabfold.batch"]), fasta_file, output_dir,
                               "--model-type", ("alphafold2_multimer_v3" if is_multimer else "alphafold2_ptm"),
                               "--msa-mode","single_sequence", "--num-models","1","--num-recycle","2",
                               "--pair-mode","paired" if is_multimer else "unpaired", "--rank","auto","--random-seed","42",
                               "--disable-unified-memory","--recompile-padding","1",
                               "--jobname", job_prefix, "--max-msa","64:64","--disable-cluster-profile","--stop-at-score","70"]
                        env = os.environ.copy(); env.update({"JAX_PLUGINS":"disabled","JAX_PLATFORM_NAME":"cpu",
                                                             "TF_CPP_MIN_LOG_LEVEL":"3","XLA_PYTHON_CLIENT_PREALLOCATE":"false",
                                                             "XLA_PYTHON_CLIENT_MEM_FRACTION":"0.75","TF_FORCE_GPU_ALLOW_GROWTH":"true"})
                        try:
                            return _run(cmd, timeout_sec=estimate_msa_timeout(seq_len, "single_sequence"), env=env)
                        except Exception as e:
                            return False, str(e)
                    okL, logL = last_chance_minirun(fa_path, str(out_dir), safe, is_complex)
                    job_dir = out_dir
                    res = analyze_results(job_dir)

                # AFDB fallback if still failing (monomer only)
                if res.get("status") != "success" and (not is_complex):
                    try:
                        fetch2 = smart_afdb_fetch(name, seq)
                        if fetch2:
                            url2 = fetch2.get("pdb_url") or fetch2.get("cif_url") or fetch2.get("bcif_url")
                            if url2:
                                local2 = out_dir / (safe + "_AFDB2" + Path(url2).suffix)
                                if not local2.exists():
                                    urllib.request.urlretrieve(url2, local2)
                                (st.session_state["_afdb_used"]).setdefault(name, fetch2.get("acc"))
                                if local2.suffix.lower() in [".cif",".bcif"]:
                                    outp2 = local2.with_suffix("").with_name(local2.stem + "_converted.pdb")
                                    try:
                                        _ = any_cif_to_pdb(local2, outp2) or (local2.suffix.lower()==".cif" and cif_to_pdb_text_cif(local2, outp2))
                                    except Exception:
                                        pass
                                res = analyze_results(job_dir)
                                if res.get("status") == "success":
                                    st.info("AFDB fallback used.")
                    except Exception as e:
                        st.info(f"AFDB fallback: {e}")

                results[name] = res
                ss.job_dirs[name] = str(job_dir); ss.fasta_paths[name] = fa_path

                if res.get("status") == "success":
                    best = (sorted(res["models"], key=lambda m: (m["rank"] if m["rank"] is not None else 9999, -(m["avg_plddt"] or 0))) or [None])[0]
                    if best:
                        st.success(f"✅ {name}: best={best['model_id']} (rank={best['rank'] or 'NA'} | pLDDT≈{(best['avg_plddt'] or 0):.1f})")
                        warns = validate_prediction({"avg_plddt":best.get("avg_plddt"), "ptm":best.get("ptm"), "iptm":best.get("iptm")})
                        for w in warns: st.warning(w)
                    else:
                        st.success(f"✅ {name}: structure detected")
                else:
                    tail = "\n".join((log or "").splitlines()[-20:])
                    st.error(f"❌ {name}: {res.get('message','Error')}\n\n{tail}")

                prog.progress((i+1)/len(valid))
            ss.results = results
            st.success("🎉 Finished — check Results / 3D")

# ---------- UI: Results ----------
elif current_page == "📈 Results":
    st.markdown('<div class="sub-header">Results</div>', unsafe_allow_html=True)
    mode = ss.get("view_mode","pLDDT (structures)")

    if mode.startswith("pLDDT"):
        if not ss.get("results"):
            st.info("No results.")
        else:
            st.markdown("### 🏆 Comparator (structures)")
            rows = []
            for name, res in ss["results"].items():
                if res.get("status") != "success":
                    continue
                for m in res["models"]:
                    rows.append({
                        "Sequence": name, "Model": m["model_id"], "Rank": m["rank"], "RankConf": m["ranking_conf"],
                        "Avg_pLDDT": m["avg_plddt"], "PTM": m["ptm"], "ipTM": m["iptm"],
                        "Format": m["fmt"].upper(), "File": m["file"]
                    })
            if not rows:
                st.warning("No model listable.")
            else:
                df = pd.DataFrame(rows).sort_values(by=["Sequence","Rank","Avg_pLDDT"], ascending=[True,True,False], na_position="last")
                st.dataframe(df, use_container_width=True, height=320)

                if not df.empty:
                    st.markdown("### 👁️ Open a model")
                    idx = st.selectbox(
                        "Model:", list(range(len(df))),
                        format_func=lambda i: f"{df.iloc[i]['Sequence']} • {df.iloc[i]['Model']} • {df.iloc[i]['Format']} • pLDDT≈{(df.iloc[i]['Avg_pLDDT'] or 0):.1f}",
                        index=0
                    )
                    rec = df.iloc[idx]; path = rec["File"]; fmt = "cif" if str(path).lower().endswith((".cif",".bcif")) else "pdb"
                    try:
                        txt = open(path).read()
                        cA, cB, cC, cD = st.columns(4)
                        with cA: style = st.selectbox("Style", ["Cartoon","Stick","Sphere","Line","Surface"], 0, key="cmp_style")
                        with cB:
                            scheme = st.selectbox("Color",
                                                  ["AlphaFold (4-color)","Special (blue/orange)","pLDDT (B-factor)","Spectrum","Chain"],
                                                  0, key="cmp_scheme")
                        with cC: mono = st.checkbox("Monomer only", value=True, key="cmp_mono")
                        with cD: first_model_only = st.checkbox("1st MODEL only", value=True, key="cmp_firstmodel")
                        chains = list_polymer_chains(path)
                        sel_chain = None
                        if mono:
                            sel_chain = st.selectbox("Chain to display", chains or ["A"], index=0, key="cmp_chain")
                        if first_model_only:
                            txt = keep_first_model_text(fmt, txt)
                        html = render_py3d_html(txt, fmt, style, scheme, mono_only=mono, sel_chain=sel_chain)
                        html = enhance_py3dmol_html(patch_py3dmol_html(html))
                        stamp = viewer_key("cmp3d", path=path, style=style, scheme=scheme, mono=mono, chain=sel_chain, first=first_model_only)
                        html = apply_viewer_stamp(html, stamp)
                        st.components.v1.html(html, height=680)
                        st.download_button("📥 Download", data=txt, file_name=Path(path).name,
                                           mime=("chemical/x-mmcif" if fmt=="cif" else "chemical/x-pdb"), use_container_width=True)
                        display_plddt_panels(txt, fmt, label=path, sel_chain=sel_chain if mono else None, key_prefix="cmp3d")
                    except Exception as e:
                        st.error(f"Read failed: {e}")

    else:
        st.markdown("### 🧬 Comparator (alignments by Identity)")

        # Manual import
        with st.expander("📥 Manual import (copy-paste a text block)", expanded=not ss.get("aln_import")):
            sample = ">QueryID\nSEQUENCE...\n>UniRef100_E9AV33\t391\t0.997\t8.068E-118\t0\t335\t336\t0\t335\t336\nMHAGAAAEAA..."
            txt_imp = st.text_area("Results block", height=160, placeholder=sample, key="res_imp_text")
            if st.button("🔎 Analyze & store (manual)", use_container_width=True, key="res_imp_btn"):
                data = parse_custom_text_results(txt_imp)
                if not data or not data.get("hits"):
                    st.error("Block not recognized.")
                else:
                    ss.setdefault("aln_import", {})
                    ss.aln_import[data["name"]] = data
                    st.success(f"✅ Imported: {data['name']} (hits={len(data['hits'])})")

        # Automatic import from .a3m
        with st.expander("⚙️ Automatic import (.a3m from your predictions)"):
            if not ss.get("job_dirs"):
                st.info("No prediction job found.")
            else:
                seqs_avail = list(ss.job_dirs.keys())
                sel_seq = st.selectbox("Predicted sequence:", seqs_avail, index=0, key="res_auto_seq")
                job_dir = Path(ss.job_dirs.get(sel_seq,""))
                if st.button("Build from .a3m", use_container_width=True, key="res_build_a3m"):
                    if not job_dir.exists():
                        st.error("job_dir not found.")
                    else:
                        with st.spinner("Scanning for .a3m / .a3m.gz…"):
                            ds = _build_auto_from_a3m(sel_seq, job_dir)
                        if not ds or not ds.get("hits"):
                            st.warning("No usable .a3m parsed for this job (unexpected format).")
                        else:
                            ss.setdefault("aln_import", {})
                            ss.aln_import[ds["name"]] = ds
                            st.success(f"✅ Automatic import: {ds['name']} (hits={len(ds['hits'])})")
                            try: st.rerun()
                            except Exception: pass

        # Hit table
        rows=[]; hits_index={}
        for name, data in (ss.get("aln_import") or {}).items():
            for h in data.get("hits", []):
                key=f"{name}::{h.get('acc','NA')}"
                rows.append({
                    "Sequence":name,
                    "Accession":h.get("acc"),
                    "Identity_%":h.get("Identity_%"),
                    "Score":h.get("Score"),
                    "Evalue":h.get("Evalue"),
                    "Cov_query_%":h.get("Cov_query_%"),
                    "Aligned_core":h.get("Aligned_core"),
                })
                hits_index[key]=(name, h, data)

        df_hits = pd.DataFrame(rows) if rows else pd.DataFrame(columns=["Sequence","Accession","Identity_%","Score","Evalue","Cov_query_%","Aligned_core"])
        if df_hits.empty:
            st.info("No alignments available.")
        else:
            with st.expander("⚙️ Display options", expanded=True):
                c1, c2, c3 = st.columns(3)
                with c1:
                    max_rows = st.slider("Number of displayed rows (top N)", 50, 2000, min(300, len(df_hits)), step=50)
                with c2:
                    enrich = st.checkbox("Enrich UniProt/AFDB (can be slow)", value=False)
                with c3:
                    sort_field = st.selectbox("Sort by", ["Identity_%","Score","Evalue","Cov_query_%","Aligned_core"], index=0)

            ascending_map = {"Evalue": True}
            asc = ascending_map.get(sort_field, False)
            df_hits = df_hits.sort_values(by=[sort_field, "Identity_%"], ascending=[asc, False], na_position="last")
            df_view = df_hits.head(max_rows).reset_index(drop=True)

            if enrich:
                with st.spinner("UniProt/AFDB enrichment…"):
                    df_view = annotate_df_with_uniprot_info(df_view, acc_col="Accession")

            try:
                st.dataframe(
                    df_view, use_container_width=True, height=340,
                    column_config={
                        "UniProt_URL": st.column_config.LinkColumn("UniProt_URL", display_text="Open UniProt"),
                        "AFDB_URL": st.column_config.LinkColumn("AFDB_URL", display_text="Open AFDB")
                    }
                )
            except Exception:
                st.dataframe(df_view, use_container_width=True, height=340)

            # 3D: Query vs AFDB hit (side-by-side)
            with st.expander("3D: Query vs AFDB hit", expanded=False):
                if df_view.empty:
                    st.info("No hits to display.")
                else:
                    idx3d = st.selectbox("Select hit", list(range(len(df_view))),
                                         format_func=lambda i: f"{df_view.iloc[i]['Sequence']} • {df_view.iloc[i]['Accession']} • Id={df_view.iloc[i]['Identity_%'] or 0:.1f}%",
                                         index=0, key="id3d_hit_idx")
                    row = df_view.iloc[idx3d]
                    seq_label = str(row["Sequence"])
                    acc_raw = str(row["Accession"])
                    # Find best query model
                    best_q = None
                    for _n, _res in (ss.get("results") or {}).items():
                        if (_res or {}).get("status") != "success":
                            continue
                        if _n in seq_label or seq_label in _n:
                            models = _res.get("models") or []
                            if models:
                                best_q = sorted(models, key=lambda m: (m["rank"] if m["rank"] is not None else 9999, -(m["avg_plddt"] or 0)))[0]
                                break
                    if not best_q:
                        choices = []
                        for _n, _res in (ss.get("results") or {}).items():
                            if (_res or {}).get("status") != "success":
                                continue
                            models = _res.get("models") or []
                            if models:
                                m0 = sorted(models, key=lambda m: (m["rank"] if m["rank"] is not None else 9999, -(m["avg_plddt"] or 0)))[0]
                                choices.append((_n, m0))
                        if not choices:
                            st.warning("No predicted query model available.")
                        else:
                            i_alt = st.selectbox("Choose query model", list(range(len(choices))),
                                                 format_func=lambda i: f"{choices[i][0]} • {choices[i][1]['model_id']} • pLDDT≈{choices[i][1]['avg_plddt'] or 0:.1f}",
                                                 index=0, key="id3d_q_alt")
                            best_q = choices[i_alt][1]
                    # Resolve AFDB for hit
                    import tempfile
                    acc_canon = _extract_uniprot_acc_from_any(acc_raw) if '_extract_uniprot_acc_from_any' in globals() else None
                    fetch = smart_afdb_fetch(acc_canon or acc_raw, None)
                    afdb_path = None
                    try:
                        if fetch:
                            url = fetch.get("pdb_url") or fetch.get("cif_url") or fetch.get("bcif_url")
                            if url:
                                tpath = Path(tempfile.gettempdir())/f"afdb_{(acc_canon or acc_raw)}{Path(url).suffix}"
                                if not tpath.exists():
                                    urllib.request.urlretrieve(url, tpath)
                                if tpath.suffix.lower() in (".cif",".bcif"):
                                    outp = tpath.with_suffix("").with_name(tpath.stem + "_converted.pdb")
                                    try:
                                        ok = any_cif_to_pdb(tpath, outp) or (tpath.suffix.lower()==".cif" and cif_to_pdb_text_cif(tpath, outp))
                                        afdb_path = str(outp if ok else tpath)
                                    except Exception:
                                        afdb_path = str(tpath)
                                else:
                                    afdb_path = str(tpath)
                    except Exception as e:
                        st.info(f"AFDB fetch: {e}")
                    cL, cR = st.columns(2)
                    with cL:
                        st.markdown("Query (best model)")
                        try:
                            qtxt = open(best_q["file"]).read()
                            fmtq = "cif" if best_q["file"].lower().endswith((".cif",".bcif")) else "pdb"
                            htmlQ = render_py3d_html(qtxt, fmtq, "Cartoon", "AlphaFold (4-color)", mono_only=True, sel_chain=None, dark_bg=False)
                            st.components.v1.html(htmlQ, height=520)
                        except Exception as e:
                            st.info(f"Query viewer: {e}")
                    with cR:
                        st.markdown("AFDB (selected hit)")
                        if not afdb_path:
                            st.info("No AFDB model resolved for this hit.")
                        else:
                            try:
                                htxt = open(afdb_path).read()
                                fmth = "cif" if afdb_path.lower().endswith((".cif",".bcif")) else "pdb"
                                htmlH = render_py3d_html(htxt, fmth, "Cartoon", "AlphaFold (4-color)", mono_only=True, sel_chain=None, dark_bg=False)
                                st.components.v1.html(htmlH, height=520)
                            except Exception as e:
                                st.info(f"AFDB viewer: {e}")

# ---------- UI: 3D Visualization ----------
elif current_page == "👁️ 3D Visualization":
    st.markdown('<div class="sub-header">👁️ 3D Visualization</div>', unsafe_allow_html=True)
    mode = ss.get("view_mode", "pLDDT (structures)")

    if mode.startswith("pLDDT"):
        if not ss.results:
            st.info("No structures.")
        else:
            # Liste des modèles disponibles
            choices = []
            for n, res in ss.results.items():
                if res.get("status") != "success":
                    continue
                for m in res["models"]:
                    label = f"{n} • {m['model_id']} • {m['fmt'].upper()} • pLDDT≈{(m['avg_plddt'] or 0):.1f}"
                    choices.append((label, m))

            if not choices:
                st.warning("No PDB/CIF.")
            else:
                idx = st.selectbox(
                    "Model:", list(range(len(choices))),
                    format_func=lambda i: choices[i][0], index=0, key="v3d_sel"
                )
                m = choices[idx][1]
                path = m["file"]
                fmt = "cif" if path.lower().endswith((".cif", ".bcif")) else "pdb"
                txt = open(path, "r").read()

                # Contrôles viewer
                c1, c2, c3, c4 = st.columns(4)
                with c1:
                    style = st.selectbox("Style", ["Cartoon", "Stick", "Sphere", "Line", "Surface"], 0, key="v3d_style")
                with c2:
                    scheme = st.selectbox(
                        "Color",
                        ["AlphaFold (4-color)", "Special (blue/orange)", "pLDDT (B-factor)", "Spectrum", "Chain"],
                        0, key="v3d_scheme"
                    )
                with c3:
                    mono = st.checkbox("Monomer only", value=True, key="v3d_mono")
                with c4:
                    first_model_only = st.checkbox("1st MODEL only", value=True, key="v3d_firstmodel")

                chains = list_polymer_chains(path)
                sel_chain = None
                if mono:
                    sel_chain = st.selectbox("Chain", chains or ["A"], index=0, key="v3d_chain")

                if first_model_only:
                    txt = keep_first_model_text(fmt, txt)

                html = render_py3d_html(txt, fmt, style, scheme, mono_only=mono, sel_chain=sel_chain)
                html = enhance_py3dmol_html(patch_py3dmol_html(html))
                stamp = viewer_key("v3d", path=path, style=style, scheme=scheme, mono=mono, chain=sel_chain, first=first_model_only)
                html = apply_viewer_stamp(html, stamp)
                st.components.v1.html(html, height=680)

                st.download_button(
                    "📥 Download", data=txt, file_name=Path(path).name,
                    mime=("chemical/x-mmcif" if fmt == "cif" else "chemical/x-pdb"),
                    use_container_width=True
                )

                # Panneaux pLDDT
                display_plddt_panels(txt, fmt, label=path, sel_chain=sel_chain if mono else None, key_prefix="v3d")

                # Domains overlay
                st.subheader("Domains (UniProt/InterPro)")
                seq_name_guess = find_seq_name_by_model_path(path) or ""
                acc_auto = auto_guess_uniprot_acc_for_sequence(seq_name_guess)
                colS, colA, colB = st.columns([1, 2, 1])
                with colS:
                    src_domains = st.radio("Source", ["UniProt", "InterPro"], horizontal=True, key="v3d_dom_src")
                with colA:
                    up_acc_in = st.text_input("UniProt Accession", value=acc_auto or "", key="v3d_up_domains")
                with colB:
                    auto_go = st.checkbox("Auto (generate if ACC found)", value=True, key="v3d_auto_dom")
                use_lbl_seq = st.checkbox("Use label_seq_id mapping (CIF only)", value=False, key="v3d_dom_lblseq")
                btn = st.button("Generate domains", key="v3d_gen_domains")
                doit = btn or (auto_go and (up_acc_in or acc_auto))
                if doit:
                    acc_use = up_acc_in or acc_auto
                    if acc_use:
                        if src_domains == "UniProt":
                            with st.spinner("Fetching domains (UniProt)…"):
                                j = fetch_uniprot_json(acc_use)
                                segs = domains_from_uniprot_json(j) if j else []
                        else:
                            with st.spinner("Fetching domains (InterPro)…"):
                                j = fetch_interpro_json(acc_use)
                                segs = interpro_segments_from_json(j) if j else []
                        if segs:
                            htmlD = render_domains_py3d_html(
                                txt, fmt, segs, style=style,
                                chain=(sel_chain if mono else None),
                                use_label_seq_id=(use_lbl_seq if fmt == "cif" else False)
                            )
                            htmlD = patch_py3dmol_html(htmlD)
                            stampD = viewer_key(
                                "v3d-dom", path=path, style=style,
                                scheme=("domains_interpro" if src_domains == "InterPro" else "domains_uniprot"),
                                mono=mono, chain=sel_chain, first=first_model_only
                            )
                            htmlD = apply_viewer_stamp(htmlD, stampD)
                            st.components.v1.html(htmlD, height=680)
                            st.markdown(domain_legend_html(segs), unsafe_allow_html=True)
                        else:
                            st.info(f"No domain found for {src_domains} ({acc_use}).")
                    else:
                        st.info("Could not auto-detect a UniProt Accession; please enter it manually if possible.")

                # PAE Heatmap (if available)
                try:
                    seq_name_for_model = find_seq_name_by_model_path(path)
                    if seq_name_for_model and (ss.results.get(seq_name_for_model) or {}).get("pae_json"):
                        with st.expander("PAE Heatmap (Predicted Aligned Error)", expanded=False):
                            fig_pae = render_pae_heatmap(ss.results[seq_name_for_model]["pae_json"], title=f"PAE — {seq_name_for_model}")
                            if fig_pae:
                                st.plotly_chart(fig_pae, use_container_width=True)
                            else:
                                st.info("PAE not available or unreadable.")
                except Exception:
                    pass

                # TM-align / US-align pane
                with st.expander("Structural alignment (TM-align / US-align)", expanded=False):
                    # Liste globale de tous les modèles (AFDB/CF) pour choisir ref/mobile
                    all_items = []
                    for _n, _res in (ss.get("results") or {}).items():
                        if (_res or {}).get("status") != "success":
                            continue
                        for _m in (_res.get("models") or []):
                            lab = f"{_n} • {_m.get('model_id','-')} • {_m.get('fmt','').upper()} • pLDDT≈{(_m.get('avg_plddt') or 0):.1f}"
                            all_items.append((lab, _m))

                    if not all_items:
                        st.info("No models available.")
                    else:
                        import tempfile
                        def _ensure_pdb(path_str: str) -> str:
                            p = Path(path_str)
                            if p.suffix.lower() == ".pdb":
                                return str(p)
                            tmp = Path(tempfile.gettempdir()) / f"usal_{p.stem}_converted.pdb"
                            try:
                                ok = any_cif_to_pdb(p, tmp) or (p.suffix.lower() == ".cif" and cif_to_pdb_text_cif(p, tmp))
                                return str(tmp if ok else p)
                            except Exception:
                                return str(p)

                        cR, cM = st.columns(2)
                        with cR:
                            i_ref = st.selectbox("Reference", list(range(len(all_items))),
                                                 format_func=lambda i: all_items[i][0], index=0, key="usal_ref")
                        with cM:
                            i_mob = st.selectbox("Mobile", list(range(len(all_items))),
                                                 format_func=lambda i: all_items[i][0], index=min(1, len(all_items)-1), key="usal_mob")

                        if i_ref == i_mob:
                            st.warning("Choose two different models.")
                        else:
                            ref = all_items[i_ref][1]
                            mob = all_items[i_mob][1]
                            pdb_ref = _ensure_pdb(ref["file"])
                            pdb_mob = _ensure_pdb(mob["file"])

                            if st.button("Align (US-align)", use_container_width=True, key="usal_go"):
                                with st.spinner("Running US-align…"):
                                    okA, logA, tm, rmsd, lali, sup_pdb = _run_usalign(pdb_ref, pdb_mob, tempfile.gettempdir())

                                # IMPORTANT: l'else est aligné avec le if; le with se termine avant
                                if not okA or not sup_pdb:
                                    st.error("US-align failed.")
                                    st.caption((logA or "")[-1000:])
                                else:
                                    st.success(
                                        f"TM-score={tm if tm is not None else 'NA'} | "
                                        f"RMSD={rmsd if rmsd is not None else 'NA'} Å | "
                                        f"Aligned length={lali if lali is not None else 'NA'}"
                                    )
                                    try:
                                        import py3Dmol
                                        v = py3Dmol.view(width=1000, height=650)
                                        # Référence en bleu foncé
                                        with open(pdb_ref, "r") as _f:
                                            v.addModel(_f.read(), "pdb")
                                        v.setStyle({"model": 0}, {"cartoon": {"color": "#1F57F7"}})
                                        # Mobile superposé en orange
                                        with open(sup_pdb, "r") as _f:
                                            v.addModel(_f.read(), "pdb")
                                        v.setStyle({"model": 1}, {"cartoon": {"color": "#FF7F0E"}})
                                        v.setBackgroundColor("white")
                                        v.zoomTo(); v.render()
                                        st.components.v1.html(v._make_html(), height=680)

                                        # Export du PDB superposé
                                        try:
                                            sup_txt = Path(sup_pdb).read_text()
                                            st.download_button(
                                                "📥 Download superposed PDB",
                                                data=sup_txt,
                                                file_name=Path(sup_pdb).name,
                                                mime="chemical/x-pdb",
                                                use_container_width=True
                                            )
                                        except Exception:
                                            pass
                                    except Exception as e:
                                        st.info(f"Viewer error: {e}")
    else:
        st.info("Use Results (Identity Mode) to inspect alignments and open AFDB/UniProt links.")
# ---------- UI: Settings ----------
elif current_page == "⚙️ Settings":
    st.markdown('<div class="sub-header">⚙️ Settings & System</div>', unsafe_allow_html=True)
    c1, c2 = st.columns(2)
    with c1:
        try:
            st.metric("CPU Cores", psutil.cpu_count())
            st.metric("Total RAM", f"{psutil.virtual_memory().total/(1024**3):.1f} GB")
        except Exception:
            st.warning("CPU/RAM info unavailable.")
        try:
            r = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"], capture_output=True, text=True)
            gpu = r.stdout.strip().split("\n")[0] if r.returncode==0 and r.stdout.strip() else "Not detected"
        except Exception:
            gpu = "Not detected"
        st.metric("GPU", gpu)
    with c2:
        try:
            import jax
            st.caption(f"JAX backend: {jax.default_backend()} | devices: {jax.devices()}")
        except Exception:
            st.caption("JAX backend: unknown")

# ---------- Footer ----------
st.markdown("---")
st.markdown('<div style="text-align:center;color:#666;padding:0.8rem 0;">🧬 AlphaFold Fusion Premium — AF Fidelity ↑ + pLDDT/Identity + AFDB + InterPro/UniProt domains</div>', unsafe_allow_html=True)
''')
# ========= Assemblage final et lancement =========

# 1) Assembler le code
app_code = ''.join(APP)

# 2) Écrire app.py
with open("app.py","w",encoding="utf-8") as f:
    f.write(app_code)
print("✅ app.py written.")

# 3) Configuration Streamlit
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml","w",encoding="utf-8") as f:
    f.write("""
[server]
headless = true
address = "0.0.0.0"
port = 8501
enableCORS = false
enableXsrfProtection = false
[browser]
gatherUsageStats = false
[theme]
primaryColor = "#4361ee"
backgroundColor = "#ffffff"
secondaryBackgroundColor = "#f8f9fa"
textColor = "#212529"
font = "sans serif"
""")

# 4) Lancer Streamlit
print("🚀 Starting the Streamlit application…")
proc = subprocess.Popen(
    [sys.executable,"-m","streamlit","run","app.py",
     "--server.port","8501","--server.address","0.0.0.0"],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT
)
time.sleep(12)

# 5) URL Colab + fallback Cloudflared
try:
    from google.colab import output
    from IPython.display import HTML, display
    public_url = output.eval_js("google.colab.kernel.proxyPort(8501)")
    print("\n🎉 Interface ready!")
    print(f"🌐 URL: {public_url}")
    display(HTML(f"""
    <div style="padding:24px;border-radius:16px;background:linear-gradient(135deg,#4361ee 0%,#3a0ca3 100%);text-align:center;color:#fff;margin-top:16px;">
      <h2 style="margin-bottom:12px;">🧬 AlphaFold Fusion Premium</h2>
      <a href="{public_url}" target="_blank" style="display:inline-block;padding:14px 26px;background:#ff6b6b;border-radius:12px;color:white;text-decoration:none;font-weight:bold;">OPEN INTERFACE</a>
      <p style="margin-top:10px;font-family:monospace;">{public_url}</p>
    </div>"""))
except Exception as e:
    print(f"⚠️ Colab proxy unavailable: {e}")
    try:
        print("🌐 Attempting Cloudflared tunnel…")
        bin_path = "/usr/local/bin/cloudflared"
        if not Path(bin_path).exists():
            url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
            subprocess.run(["wget","-q","-O", bin_path, url], check=False)
            os.chmod(bin_path,0o755)
        proc_cf = subprocess.Popen(
            [bin_path,"tunnel","--url","http://127.0.0.1:8501","--no-autoupdate"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
        )
        tunnel_url = None
        t0 = time.time()
        import re as _re
        while time.time()-t0 < 90:
            line = proc_cf.stdout.readline()
            if not line:
                time.sleep(0.2); continue
            m = _re.search(r"https://[-\w]+\.trycloudflare\.com", line)
            if m:
                tunnel_url = m.group(0); break
        if tunnel_url:
            print("🎉 Cloudflared tunnel:", tunnel_url)
            from IPython.display import HTML, display
            display(HTML(f"""
            <div style="padding:24px;border-radius:16px;background:#0ea5e9;text-align:center;color:#fff;margin-top:16px;">
              <h2 style="margin-bottom:12px;">🧬 AlphaFold Fusion Premium</h2>
              <a href="{tunnel_url}" target="_blank" style="display:inline-block;padding:14px 26px;background:#111827;border-radius:12px;color:white;text-decoration:none;font-weight:bold;">OPEN (Cloudflared)</a>
              <p style="margin-top:10px;font-family:monospace;">{tunnel_url}</p>
            </div>"""))
        else:
            print("❌ No Cloudflared URL. If running locally: http://localhost:8501")
    except Exception as e2:
        print("❌ Cloudflared failed:", e2)
        print("If local: http://localhost:8501")

🧬 AlphaFold Fusion Premium — AF fidelity ↑ (MSA/Multimer/Recycles) + pLDDT + Identity + AFDB-first + Reliable 3D
  → colabfold[alphafold]
    → pip colabfold[alphafold] @ https://codeload.github.com/sokrypton/ColabFold/zip/refs/heads/main
    → pip numpy==1.26.4
    → pip pandas>=2,<3
    → pip tensorflow==2.18.* protobuf>=4.25,<6
    → pip streamlit==1.28.0
    → pip plotly==5.17.0
    → pip py3Dmol==2.1.0
    → pip biopython>=1.83,<2
    → pip Pillow==10.1.0
    → pip psutil==5.9.8
    → pip gemmi==0.6.6
    → pip jax>=0.5.0,<0.6 jaxlib>=0.5.0,<0.6
✅ JAX ready — backend: cpu | devices: [CpuDevice(id=0)]
⚠️ NumPy/Pandas import: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject
✅ Dependencies installed
✅ app.py written.
🚀 Starting the Streamlit application…

🎉 Interface ready!
🌐 URL: https://8501-m-s-nsfnshxryr9x-a.us-central1-0.prod.colab.dev


In [22]:
# af_fusion_mega_patch.py
import re
from pathlib import Path

APP = Path("app.py")
if not APP.exists():
    raise SystemExit("app.py introuvable — exécute d’abord le script qui génère app.py.")

src = APP.read_text(encoding="utf-8")
APP.with_suffix(".py.bak").write_text(src, encoding="utf-8")

changed = False

# =========================
# 1) Helpers gemmi 0.6.x
# =========================
helper_block = r'''
# ---------- Helpers: gemmi (0.6.x) compatibility ----------
AMINO3_STD = {
    "ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS","ILE",
    "LEU","LYS","MET","PHE","PRO","SER","THR","TRP","TYR","VAL",
    # élargis pour cas courants
    "SEC","PYL","ASX","GLX","XLE","MSE","UNK"
}

def find_atom_safe(res, name):
    """
    Version sûre pour gemmi: évite altloc='', essaie plusieurs altloc, puis fallback par itération.
    """
    for alt in (" ", "A", "B", "1", "?"):
        try:
            a = res.find_atom(name, alt)
            if a is not None:
                return a
        except Exception:
            continue
    try:
        target = str(name).strip().upper()
        for a in res:
            an = getattr(a, "name", "")
            if an and an.strip().upper() == target:
                return a
    except Exception:
        pass
    return None

def is_aa_gemmi(res) -> bool:
    """
    Heuristique robuste pour savoir si un résidu est un acide aminé (gemmi 0.6.x):
    - Nom 3 lettres standard (inclut MSE/SEC/PYL/UNK)
    - Fallback: présence d'un atome CA, ou du duo N & C (charpente)
    """
    try:
        name = (res.name or "").strip().upper()
    except Exception:
        name = ""
    if name in AMINO3_STD:
        return True
    try:
        if find_atom_safe(res, "CA") is not None:
            return True
    except Exception:
        pass
    try:
        atom_names = {getattr(a, "name", "").strip().upper() for a in res}
        if "N" in atom_names and "C" in atom_names:
            return True
    except Exception:
        pass
    return False
'''.lstrip("\n")

def insert_helpers(text: str) -> tuple[str, bool]:
    if "def find_atom_safe(" in text and "def is_aa_gemmi(" in text:
        return text, False

    # Idéal: avant list_polymer_chains
    m = re.search(r"\ndef\s+list_polymer_chains\s*\(", text)
    if m:
        pos = m.start()
        return text[:pos] + "\n" + helper_block + text[pos:], True

    # Sinon: avant af_plddt_per_residue_by_chain_from_text
    m = re.search(r"\ndef\s+af_plddt_per_residue_by_chain_from_text\s*\(", text)
    if m:
        pos = m.start()
        return text[:pos] + "\n" + helper_block + text[pos:], True

    # Sinon: après import py3Dmol, psutil, gemmi
    m = re.search(r"\nimport\s+py3Dmol,\s*psutil,\s*gemmi[^\n]*\n", text)
    if m:
        pos = m.end()
        return text[:pos] + "\n" + helper_block + text[pos:], True

    # Sinon: après bloc d'import
    m = re.search(r"(?:^|\n)(?:from\s+\S+\s+import[^\n]*\n|import\s+\S+[^\n]*\n)+", text)
    if m:
        pos = m.end()
        return text[:pos] + "\n" + helper_block + text[pos:], True

    # Ultime: en tête
    return helper_block + "\n" + text, True

def replace_is_amino_acid(text: str) -> tuple[str, int]:
    # any_identifier.is_amino_acid() -> is_aa_gemmi(any_identifier)
    pat = re.compile(r"(\b[A-Za-z_][A-Za-z0-9_]*)\.is_amino_acid\(\)")
    return pat.subn(r"is_aa_gemmi(\1)", text)

def replace_find_atom_empty_altloc(text: str) -> tuple[str, int]:
    # x.find_atom('CA','') -> find_atom_safe(x,'CA')
    pat = re.compile(
        r"(\b[A-Za-z_][A-Za-z0-9_\.]*?)\.find_atom\(\s*(['\"][^'\"]+['\"])\s*,\s*(?:\"\"|'')\s*\)"
    )
    return pat.subn(r"find_atom_safe(\1, \2)", text)

src, helpers_added = insert_helpers(src)
if helpers_added:
    changed = True

src, n_isaa = replace_is_amino_acid(src)
if n_isaa > 0:
    changed = True

src, n_find = replace_find_atom_empty_altloc(src)
if n_find > 0:
    changed = True

# =========================
# 2) OOM GPU safe
# =========================

# 2a) run_colabfold_advanced: ne pas désactiver UM si autorisée ou L>=900
def patch_unified_memory_guard(text: str) -> tuple[str, bool]:
    pat1 = re.compile(
        r'(?P<indent>\s*)if\s+_sh\.which\(\s*"nvidia-smi"\s*\)\s*:\s*cmd\s*\+\=\s*\[\s*"--disable-unified-memory"\s*\]',
        re.M
    )
    pat2 = re.compile(
        r'(?P<indent>\s*)if\s+_sh\.which\(\s*\'nvidia-smi\'\s*\)\s*:\s*cmd\s*\+\=\s*\[\s*\'--disable-unified-memory\'\s*\]',
        re.M
    )
    def repl(m):
        indent = m.group("indent")
        return (
            f'{indent}allow_um = bool(params.get("allow_unified_memory", False)) or (int(seq_len) if seq_len else 0) >= 900\n'
            f'{indent}if _sh.which("nvidia-smi") and not allow_um: cmd += ["--disable-unified-memory"]'
        )
    new_text, n1 = pat1.subn(repl, text, count=1)
    if n1 == 0:
        new_text, n2 = pat2.subn(repl, text, count=1)
        return new_text, (n2 > 0)
    return new_text, True

src, um_patched = patch_unified_memory_guard(src)
if um_patched:
    changed = True

# 2b) Insérer retry GPU OOM-safe après 1er analyze_results(job_dir)
def insert_oom_retry(text: str) -> tuple[str, bool]:
    anchor = "ok, log = run_colabfold_advanced("
    i = text.find(anchor)
    if i == -1:
        return text, False
    j = text.find("res = analyze_results(job_dir)", i)
    if j == -1:
        return text, False

    # Trouver indentation de la ligne res = analyze_results(job_dir)
    line_start = text.rfind("\n", 0, j) + 1
    line_end = text.find("\n", j)
    indent = text[line_start:line_end][:len(text[line_start:line_end]) - len(text[line_start:line_end].lstrip(" "))]
    block = (
        "\n"
        f"{indent}# OOM-safe GPU retry\n"
        f'{indent}if (res.get("status") != "success") and any(s in (log or "") for s in ("RESOURCE_EXHAUSTED","Out of memory","ran out of memory")):\n'
        f'{indent}    st.info("🛟 GPU OOM detected — retrying with Unified Memory + minimal settings…")\n'
        f"{indent}    params_oom = dict(params)\n"
        f"{indent}    params_oom.update({{\n"
        f'{indent}        "allow_unified_memory": True,\n'
        f'{indent}        "num_models": 1,\n'
        f'{indent}        "num_recycles": min(int(params.get("num_recycles", 3)), 2),\n'
        f'{indent}        "use_amber": False,\n'
        f'{indent}        "use_templates": False,\n'
        f'{indent}        "msa_strategy": "minimal",\n'
        f'{indent}        "pair_mode": ("paired" if is_complex else "unpaired"),\n'
        f'{indent}        "stop_at_score": 70.0,\n'
        f'{indent}        "_max_msa": "32:32",\n'
        f"{indent}    }})\n"
        f"{indent}    ok2, log2 = run_colabfold_advanced(fa_path, str(out_dir), params_oom, seq_len=seq_len)\n"
        f"{indent}    job_dir = out_dir\n"
        f"{indent}    res = analyze_results(job_dir)\n"
    )
    insert_pos = line_end + 1
    return text[:insert_pos] + block + text[insert_pos:], True

src, retry_added = insert_oom_retry(src)
if retry_added:
    changed = True

# 2c) Nettoyer fallback CPU: enlever --disable-unified-memory et augmenter timeout
if '"--disable-unified-memory",' in src or "'--disable-unified-memory'," in src:
    src = src.replace('"--disable-unified-memory",', "")
    src = src.replace("'--disable-unified-memory',", "")
    changed = True

pat_cpu_to = re.compile(
    r'_run\(\s*cmd\s*,\s*timeout_sec\s*=\s*estimate_msa_timeout\(\s*seq_len\s*,\s*"single_sequence"\s*\)\s*,\s*env\s*=\s*env\s*\)'
)
src_new = pat_cpu_to.sub(
    r'_run(cmd, timeout_sec=max(7200, estimate_msa_timeout(seq_len, "single_sequence")*2), env=env)',
    src, count=1
)
if src_new != src:
    src = src_new
    changed = True

# =========================
# Écriture
# =========================
if changed:
    APP.write_text(src, encoding="utf-8")
    print("✅ Mega patch appliqué avec succès:")
    print("   • Helpers gemmi: find_atom_safe(), is_aa_gemmi()")
    print(f"   • Remplacements is_amino_acid() → is_aa_gemmi(): {n_isaa}")
    print(f"   • Remplacements .find_atom(..., \"\") → find_atom_safe(...): {n_find}")
    print(f"   • UM guard patché: {um_patched}")
    print(f"   • Retry OOM ajouté: {retry_added}")
    print("   • Fallback CPU: UM retiré et timeout augmenté")
    print(f"   • Sauvegarde: {APP.with_suffix('.py.bak').name}")
else:
    print("ℹ️ Rien à modifier (patch déjà appliqué ou code différent).")

✅ Mega patch appliqué avec succès:
   • Helpers gemmi: find_atom_safe(), is_aa_gemmi()
   • Remplacements is_amino_acid() → is_aa_gemmi(): 6
   • Remplacements .find_atom(..., "") → find_atom_safe(...): 2
   • UM guard patché: True
   • Retry OOM ajouté: True
   • Fallback CPU: UM retiré et timeout augmenté
   • Sauvegarde: app.py.bak
